# AI Shield — fine-tune the original full pipeline
**Integrated fixes v2 — semantic compatibility and setup recovery are built in.**
**Goal:** find out whether your original forensic + semantic system improves after adaptation
to the same Dataset v1 used for Community Forensics (CF).

| Component | What this experiment does |
|---|---|
| EfficientNet-B0 forensic model | Start from your saved weights; fine-tune its head, then its final block |
| ViT + your semantic attention head | Start from your saved head and original backbone; fine-tune the head, then the final block |
| VAE reconstruction + face/OpenForensics signals | Keep the weights frozen; calculate these signals once |
| EfficientNet calibration | Refit the score conversion using a separate training subset |
| Full fusion | Refit the original **17-feature** combination, including auxiliary signals |
| Community Forensics | Keep the existing detector fixed as a comparison |

This extends your saved system, including your custom components and pretrained backbones.
It does **not** train every model from scratch. EXIF, C2PA and the planned context layer are outside this experiment.

We compare **original saved full pipeline → frozen networks with refitted fusion → fine-tuned full pipeline**,
plus the original and fine-tuned CF results. Accuracy improvement is a question to test, not a promised result.

**To run:** choose a fresh Colab **GPU runtime (Python 3.11–3.13)**, then run cells in order.
The Drive mount will ask you to authorize access. Use a fresh session if packages have already been imported.
This notebook has been generated and checked locally; the full Drive/GPU experiment has not run yet.
Original checkpoints and the application detector are read-only. Section 3 reuses this session's
experiment or restores saved setup; it creates a folder only when no existing experiment is selected.
**Existing session:** run section 2, then 3, then 4. Do not run the old diagnostic cells at the bottom.
**New runtime:** install section 1 first, follow its restart instruction if shown, then run from section 2.
Setup recovery is included. Resuming an interrupted training optimizer/epoch is not yet implemented.


## 1. Install the experiment's dependencies
Uses Colab's compatible PyTorch/torchvision pair. TensorFlow runs the frozen face model on CPU.
Supports the current Python 3.13 runtime using TensorFlow 2.20. The previous 2.19 pin did not.
First resolves compatible binary packages without changing the runtime; only then installs them.
Only this temporary runtime's overlapping TensorFlow/OpenCV installations are replaced.
Unused JAX is removed here to avoid conflicting TensorFlow/XLA libraries.
If Colab asks for a restart, restart once and continue from section 2 before importing these packages.


In [ ]:
PACKAGES = ['timm==1.0.15', 'transformers==4.57.6', 'scikit-learn==1.6.1', 'pandas==2.2.3', 'numpy==2.1.3', 'Pillow==11.3.0', 'psutil>=5.9,<8', 'tqdm>=4.66,<5', 'joblib>=1.3,<2', 'tensorflow-cpu==2.20.0', 'keras==3.10.0', 'opencv-python-headless==4.11.0.86', 'protobuf>=5.28.0,<7']

import subprocess, sys, os, json, importlib.metadata
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12), (3, 13)):
    raise RuntimeError('This setup supports Python 3.11, 3.12 and 3.13. No packages were changed.')
print('Python:', sys.version.split()[0], flush=True)
loaded_before = [name for name in ('numpy', 'pandas', 'sklearn', 'PIL', 'tensorflow', 'keras', 'cv2')
                 if name in sys.modules]
# Keep Colab's CUDA-enabled torch/torchvision pair exactly as installed.
constraints = Path('/content/ai_shield_torch_constraints.txt')
constraints.write_text(''.join(name + '==' + importlib.metadata.version(name) + chr(10)
                               for name in ('torch', 'torchvision')))
install_options = ['--only-binary=:all:', '--constraint', str(constraints)]
log_path = Path('/content/ai_shield_setup.log')
log_path.write_text('Python ' + sys.version + chr(10))
def run_pip(arguments, stage):
    print(stage + '...', flush=True)
    result = subprocess.run([sys.executable, '-m', 'pip', *arguments],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with log_path.open('a') as log:
        log.write(stage + chr(10) + result.stdout + chr(10))
    if result.returncode:
        print(result.stdout[-16000:])
        raise RuntimeError(stage + ' failed. The actual pip error is printed above; full log: ' + str(log_path))
    print(stage + ': OK', flush=True)
run_pip(['install', '--dry-run', *install_options, *PACKAGES], 'Checking dependency compatibility')
run_pip(['uninstall', '-y',
    'tensorflow', 'tensorflow-cpu', 'tf-nightly', 'tf-nightly-cpu', 'jax', 'jaxlib',
    'opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'],
    'Replacing overlapping runtime packages')
run_pip(['install', *install_options, *PACKAGES], 'Installing compatible packages')
# A separate interpreter verifies imports without loading changed libraries into this kernel.
smoke = subprocess.run([sys.executable, '-c',
    'import numpy,pandas,sklearn,cv2,torch,torchvision,timm; '
    'from transformers import ViTModel; import tensorflow as tf; '
    'print("IMPORT CHECK PASSED", "TensorFlow", tf.__version__, "PyTorch", torch.__version__); '
    'assert tf.reduce_sum(tf.constant([1.,2.])).numpy() == 3.'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with log_path.open('a') as log:
    log.write(smoke.stdout)
if smoke.returncode:
    print(smoke.stdout[-16000:])
    raise RuntimeError('Package import check failed; see the actual error above.')
print(smoke.stdout[-2000:])
if loaded_before:
    Path('/content/ai_shield_restart_required.json').write_text(json.dumps({'pid': os.getpid()}))
    print('INSTALLATION COMPLETE. Restart the Colab session once, then continue at section 2.')
else:
    print('INSTALLATION COMPLETE. Continue at section 2.')


## 2. Load the code
All required research code is embedded below; no GitHub push is needed.


In [ ]:
EMBEDDED_SOURCES = {'src/ai_shield_adaptation.py': '"""Controlled two-branch adaptation experiment. Never imported by the app."""\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport random\nimport time\nimport uuid\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import precision_recall_curve\nfrom sklearn.model_selection import GroupShuffleSplit\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom src.ai_shield_metrics import binary_metrics\n\n\nEXPECTED_COUNTS = {"train": 2990, "validation": 640, "test": 640, "unseen_generator": 854}\nSPLIT_HASHES = {\n    "train": "29e893703856c3e1c5e71a0a5f04619ef01bb0e7cc8829a01e39ce605fc9f2a2",\n    "validation": "1d69e4c55ab1896b51421b2e9bbc3d1a919966087ca7601f8f3f3919250aaf11",\n    "test": "fa53c8831a4d4657d1ea34558b284d13dc65935686255761f89638c6802af063",\n    "unseen_generator": "3fc0d4ea61bab0340b9512e732042bddb3f066768aadb9c1dbc574c05fb8d392",\n}\nSEMANTIC_ID = "google/vit-base-patch16-224-in21k"\nSEMANTIC_REVISION = "b4569560a39a0f1af58e3ddaf17facf20ab919b0"\nFEATURES = [\n    "effnet_logit", "effnet_prob", "semantic_vit_logit", "semantic_vit_prob",\n    "semantic_attention_entropy", "semantic_attention_focus",\n    "semantic_self_attention_entropy", "semantic_self_attention_focus",\n]\n\n\n@dataclass(frozen=True)\nclass Config:\n    drive_root: str = "/content/drive/MyDrive"\n    seed: int = 191\n    fusion_fraction: float = 0.20\n    head_epochs: int = 3\n    partial_epochs: int = 3\n    head_lr: float = 3e-4\n    partial_head_lr: float = 1e-4\n    backbone_lr: float = 1e-5\n    weight_decay: float = 1e-2\n    batch_size: int = 16\n    num_workers: int = 0\n\n    @property\n    def root(self):\n        return Path(self.drive_root)\n\n    @property\n    def split_root(self):\n        return self.root / "AI_Shield_Data/splits/v1"\n\n    @property\n    def cf_root(self):\n        return self.root / "AI_Shield_CommunityForensics/experiments/finetune_v1"\n\n    @property\n    def audit_root(self):\n        return self.root / "AI_Shield_Baseline_Evaluation/saved_pipeline_v1/results"\n\n    @property\n    def checkpoints(self):\n        return {\n            "effnet": self.root / "AI_Shield_Phase1/effnet_b0_tinygenimage_full_balanced_v1/effnet_b0_tinygenimage_full_balanced_v1_best.pt",\n            "semantic": self.root / "AI_Shield_Phase2_Semantic/semantic_vit_transformer_attention_v1/checkpoints/semantic_vit_transformer_attention_best.pt",\n        }\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef write_json(path, payload):\n    # Exclusive creation makes a repeated cell fail instead of replacing evidence.\n    with Path(path).open("x", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2, allow_nan=False)\n        handle.write("\\n")\n\n\ndef checked_csv(path):\n    return pd.read_csv(path, keep_default_na=False, dtype={\n        "sample_id": str, "sha256": str, "group_id": str, "prompt_id": str,\n    })\n\n\ndef validate_frames(frames, expected_counts=None):\n    if set(frames) != set(EXPECTED_COUNTS):\n        raise ValueError("Expected train, validation, test and unseen_generator splits.")\n    required = {"sample_id", "label", "split", "sha256", "group_id", "prompt_id", "generator", "local_path"}\n    for name, frame in frames.items():\n        if not required.issubset(frame):\n            raise ValueError(f"{name}: missing columns {required - set(frame)}")\n        if expected_counts and len(frame) != expected_counts[name]:\n            raise ValueError(f"{name}: wrong sample count")\n        if set(frame.label) != {0, 1} or set(frame.split) != {name}:\n            raise ValueError(f"{name}: invalid labels or split values")\n        for column in ("sample_id", "group_id", "sha256", "local_path"):\n            if frame[column].isna().any() or frame[column].astype(str).str.strip().eq("").any():\n                raise ValueError(f"{name}: empty {column}")\n        if not frame.sha256.str.fullmatch(r"[a-f0-9]{64}").all():\n            raise ValueError(f"{name}: malformed SHA-256")\n    all_rows = pd.concat(frames.values(), ignore_index=True)\n    if all_rows.sample_id.duplicated().any():\n        raise ValueError("Duplicate sample IDs")\n    for column in ("sha256", "group_id", "local_path"):\n        if (all_rows.groupby(column).split.nunique() > 1).any():\n            raise ValueError(f"{column} leakage across outer splits")\n    if (all_rows.groupby("sha256").label.nunique() > 1).any():\n        raise ValueError("Identical image bytes have conflicting labels")\n    dev = all_rows[all_rows.split != "unseen_generator"]\n    fake = dev[dev.label == 1]\n    if fake.generator.eq("gpt-image-2").any():\n        raise ValueError("Held-out generator entered development")\n    prompts = fake[fake.prompt_id.astype(str).str.strip().ne("")]\n    if (prompts.groupby("prompt_id").split.nunique() > 1).any():\n        raise ValueError("Prompt leakage across development splits")\n    unseen = frames["unseen_generator"]\n    if set(unseen.loc[unseen.label == 1, "generator"]) != {"gpt-image-2"}:\n        raise ValueError("Unexpected held-out generator")\n\n\ndef partition_training(train, fraction=0.20, seed=191):\n    if set(train.split) != {"train"}:\n        raise ValueError("Inner partition must only use the original training split")\n    if not 0 < fraction < 1:\n        raise ValueError("Fusion fraction must be in (0, 1)")\n    train = train.reset_index(drop=True)\n    # Connected groups keep prompts, source groups and duplicate bytes together.\n    parent = list(range(len(train)))\n\n    def find(index):\n        while parent[index] != index:\n            parent[index] = parent[parent[index]]\n            index = parent[index]\n        return index\n\n    for column in ("group_id", "sha256", "prompt_id"):\n        seen = {}\n        for index, row in train.iterrows():\n            value = str(row[column]).strip()\n            if not value or (column == "prompt_id" and row.label != 1):\n                continue\n            if value in seen:\n                parent[find(index)] = find(seen[value])\n            else:\n                seen[value] = index\n    groups = [find(i) for i in range(len(train))]\n    splitter = GroupShuffleSplit(n_splits=1, test_size=fraction, random_state=seed)\n    branch, fusion = next(splitter.split(train, train.label, groups))\n    result = train.copy()\n    result["role"] = "branch_train"\n    result.loc[fusion, "role"] = "fusion_train"\n    for role, subset in result.groupby("role"):\n        if set(subset.label) != {0, 1}:\n            raise ValueError(f"{role} needs both classes; do not silently resplit")\n    return result\n\n\ndef align_predictions(reference, predictions, columns):\n    required = {"sample_id", "label", *columns}\n    if not required.issubset(predictions):\n        raise ValueError(f"Missing prediction columns: {required - set(predictions)}")\n    if reference.sample_id.duplicated().any() or predictions.sample_id.duplicated().any():\n        raise ValueError("Duplicate prediction IDs")\n    if set(reference.sample_id) != set(predictions.sample_id):\n        raise ValueError("Prediction IDs differ from the manifest; no partial comparisons")\n    ordered = predictions.set_index("sample_id").loc[reference.sample_id].reset_index()\n    if not np.array_equal(reference.label.to_numpy(), ordered.label.to_numpy()):\n        raise ValueError("Prediction labels do not match manifest")\n    if not np.isfinite(ordered[columns].to_numpy(dtype=float)).all():\n        raise ValueError("Non-finite predictions")\n    return ordered\n\n\ndef max_f1_threshold(labels, scores):\n    labels, scores = np.asarray(labels), np.asarray(scores, dtype=float)\n    if set(labels) != {0, 1} or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n        raise ValueError("Need both classes and finite [0, 1] scores")\n    precision, recall, thresholds = precision_recall_curve(labels, scores)\n    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-15)\n    return float(thresholds[np.flatnonzero(f1 == f1.max())[-1]])\n\n\ndef fit_fusion(predictions, branch_training_ids):\n    if set(predictions.split) != {"train"}:\n        raise ValueError("Fusion fitting is restricted to training images")\n    if set(predictions.sample_id) & set(branch_training_ids):\n        raise ValueError("Fusion images were used to fine-tune a branch")\n    align_predictions(predictions, predictions, FEATURES)\n    if set(predictions.label) != {0, 1}:\n        raise ValueError("Fusion needs both classes")\n    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=191))\n    model.fit(predictions[FEATURES], predictions.label)\n    return model\n\n\ndef preflight(config):\n    frames = {}\n    for name in EXPECTED_COUNTS:\n        path = config.split_root / f"{name}.csv"\n        if sha256_file(path) != SPLIT_HASHES[name]:\n            raise ValueError(f"{name}: differs from the recorded CF split")\n        frames[name] = checked_csv(path)\n    validate_frames(frames, EXPECTED_COUNTS)\n    audit = json.loads((config.audit_root / "saved_pipeline_evaluation_report.json").read_text())\n    for branch, path in config.checkpoints.items():\n        if sha256_file(path) != audit["source_artifacts"][f"{branch}_sha256"]:\n            raise ValueError(f"{branch}: checkpoint differs from the saved baseline audit")\n    from tqdm.auto import tqdm\n    for row in tqdm(pd.concat(frames.values()).itertuples(), total=sum(EXPECTED_COUNTS.values()), desc="Verify image bytes"):\n        if sha256_file(row.local_path) != row.sha256:\n            raise ValueError(f"Changed image: {row.sample_id}")\n    return frames, audit\n\n\ndef prepare_run(config, frames):\n    if config.head_epochs < 1 or config.partial_epochs < 0 or config.batch_size < 1:\n        raise ValueError("Invalid training budget")\n    partition = partition_training(frames["train"], config.fusion_fraction, config.seed)\n    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    run = config.root / "AI_Shield_Adaptation" / "two_branch_v1" / run_id\n    run.mkdir(parents=True, exist_ok=False)\n    partition.to_csv(run / "training_roles.csv", index=False)\n    versions = {name: importlib.metadata.version(name) for name in (\n        "torch", "torchvision", "timm", "transformers", "scikit-learn", "pandas", "numpy", "Pillow",\n    )}\n    write_json(run / "experiment_plan.json", {\n        "config": asdict(config), "libraries": versions, "split_sha256": SPLIT_HASHES,\n        "source_sha256": {k: sha256_file(v) for k, v in config.checkpoints.items()},\n        "implementation_sha256": sha256_file(__file__),\n        "semantic_revision": SEMANTIC_REVISION, "feature_order": FEATURES,\n        "training_roles_sha256": sha256_file(run / "training_roles.csv"),\n        "counts_by_role": partition.role.value_counts().to_dict(),\n        "scope": "Two-branch variant, NOT the original 17-feature full pipeline. No VAE/face branch.",\n        "budget_note": "Same 2990-image development budget as CF; branch training uses only the branch_train subset. CF trained on all 2990.",\n        "preprocessing": "Original audit: frame 0, RGB without EXIF transpose; EffNet resize256 bilinear/ImageNet; semantic pinned HF processor. Train-only horizontal flip 0.5.",\n        "test_policy": "Previously examined historical tests; never used for fitting/selection. Fresh confirmation still needed.",\n        "pretraining_cost": "Not measured; training times cover only this adaptation, not original pretraining.",\n    })\n    return run, partition\n\n\ndef seed_everything(seed):\n    import torch\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cudnn.benchmark = False\n\n\ndef open_rgb(path):\n    from PIL import Image\n    with Image.open(path) as image:\n        image.seek(0)\n        return image.convert("RGB").copy()\n\n\ndef make_loader(frame, branch, processor, config, training=False):\n    import torch\n    from torch.utils.data import DataLoader, Dataset\n    from torchvision import transforms\n    transform = transforms.Compose([\n        transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),\n        transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n    ])\n\n    class Images(Dataset):\n        def __len__(self):\n            return len(frame)\n\n        def __getitem__(self, index):\n            from PIL import Image\n            row = frame.iloc[index]\n            image = open_rgb(row.local_path)\n            if training and random.random() < 0.5:\n                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)\n            x = transform(image) if branch == "effnet" else processor(images=image, return_tensors="pt")["pixel_values"][0]\n            return x, float(row.label), str(row.sample_id)\n\n    # Zero workers is deliberate: the notebook-local dataset also works on Windows.\n    if config.num_workers != 0:\n        raise ValueError("This portable loader uses num_workers=0")\n    generator = torch.Generator().manual_seed(config.seed)\n    return DataLoader(Images(), batch_size=config.batch_size, shuffle=training, num_workers=0, generator=generator)\n\n\ndef predict_branch(model, loader, frame, branch, device):\n    import torch\n    model.to(device).eval()\n    rows = []\n    with torch.inference_mode():\n        for x, labels, ids in loader:\n            result = model(x.to(device))\n            logits = result if branch == "effnet" else result["logits"]\n            if not torch.isfinite(logits).all():\n                raise ValueError("Non-finite model logits")\n            scores = torch.sigmoid(logits).cpu().tolist()\n            for i, sample_id in enumerate(ids):\n                prefix = "effnet" if branch == "effnet" else "semantic_vit"\n                row = {"sample_id": sample_id, "label": int(labels[i]),\n                       f"{prefix}_logit": float(logits[i].cpu()), f"{prefix}_prob": scores[i]}\n                if branch == "semantic":\n                    for key in ("attention_entropy", "attention_focus", "self_attention_entropy", "self_attention_focus"):\n                        row[f"semantic_{key}"] = float(result[key][i].cpu())\n                rows.append(row)\n    columns = FEATURES[:2] if branch == "effnet" else FEATURES[2:]\n    result = align_predictions(frame, pd.DataFrame(rows), columns)\n    result["split"] = frame.split.to_numpy()\n    return result\n\n\ndef predict_pair(models, processor, frame, config, device):\n    results = []\n    for branch, model in models.items():\n        loader = make_loader(frame, branch, processor, config)\n        results.append(predict_branch(model, loader, frame, branch, device))\n        model.cpu()\n    merged = results[0].merge(results[1], on=["sample_id", "label", "split"], validate="one_to_one")\n    return align_predictions(frame, merged, FEATURES)\n\n\ndef check_audit_parity(models, processor, frames, config, device):\n    # This is a fixed implementation check, not a metric used for model selection.\n    fixture = frames["test"].sort_values("sample_id").groupby("label", group_keys=False).head(3)\n    actual = predict_pair(models, processor, fixture, config, device)\n    saved = checked_csv(config.audit_root / "aishield_test_predictions.csv")\n    saved = saved[saved.sample_id.isin(fixture.sample_id)]\n    saved = align_predictions(fixture, saved, FEATURES)\n    differences = {}\n    for column in FEATURES:\n        a, b = actual[column].to_numpy(), saved[column].to_numpy()\n        differences[column] = float(np.max(np.abs(a - b)))\n        if not np.allclose(a, b, atol=5e-4, rtol=1e-4):\n            raise ValueError(f"Saved baseline parity failed: {column}. Stop before training; investigate preprocessing/backbone versions.")\n    return {"n": len(fixture), "sample_ids": fixture.sample_id.tolist(), "max_absolute_errors": differences}\n\n\ndef configure_training(model, branch, stage):\n    model.requires_grad_(False)\n    if branch == "effnet":\n        head = model.model.classifier\n        backbone = [model.model.blocks[-1], model.model.conv_head, model.model.bn2]\n    else:\n        head = model.semantic_head\n        backbone = [model.vit.encoder.layer[-1], model.vit.layernorm]\n    head.requires_grad_(True)\n    if stage == "partial":\n        for module in backbone:\n            module.requires_grad_(True)\n    elif stage != "head":\n        raise ValueError("Unknown training stage")\n    return head, backbone\n\n\ndef training_mode(model, branch, stage, head, backbone):\n    # requires_grad=False alone would not freeze BatchNorm running statistics.\n    model.eval()\n    head.train()\n    if branch == "effnet":\n        model.model.training = True  # classifier\'s functional dropout only\n    if stage == "partial":\n        for module in backbone:\n            module.train()\n\n\ndef train_branch(model, branch, processor, branch_frame, validation, config, run, device):\n    import torch\n    from sklearn.metrics import roc_auc_score\n    if set(branch_frame.split) != {"train"} or set(validation.split) != {"validation"}:\n        raise ValueError("Wrong training/selection split")\n    seed_everything(config.seed)\n    checkpoint = Path(run) / f"{branch}_adapted.pt"\n    if checkpoint.exists():\n        raise FileExistsError("Start a new run instead of overwriting a trained branch")\n    model.to(device)\n    train_loader = make_loader(branch_frame, branch, processor, config, training=True)\n    val_loader = make_loader(validation, branch, processor, config)\n    score_column = "effnet_prob" if branch == "effnet" else "semantic_vit_prob"\n    initial = predict_branch(model, val_loader, validation, branch, device)\n    best_auc = float(roc_auc_score(initial.label, initial[score_column]))\n    selected = {"stage": "original", "epoch": 0, "validation_auc": best_auc}\n    torch.save({"model_state": model.state_dict(), **selected}, checkpoint)\n    history, max_trainable = [], 0\n    if str(device).startswith("cuda"):\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n    start = time.perf_counter()\n    for stage, epochs in (("head", config.head_epochs), ("partial", config.partial_epochs)):\n        if epochs == 0:\n            continue\n        model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True)["model_state"], strict=True)\n        head, backbone = configure_training(model, branch, stage)\n        head_ids = {id(p) for p in head.parameters()}\n        groups = [{"params": list(head.parameters()), "lr": config.head_lr if stage == "head" else config.partial_head_lr}]\n        others = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]\n        if others:\n            groups.append({"params": others, "lr": config.backbone_lr})\n        optimizer = torch.optim.AdamW(groups, weight_decay=config.weight_decay)\n        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)\n        max_trainable = max(max_trainable, trainable)\n        for epoch in range(1, epochs + 1):\n            training_mode(model, branch, stage, head, backbone)\n            loss_sum, count = 0.0, 0\n            epoch_start = time.perf_counter()\n            for x, labels, _ in train_loader:\n                optimizer.zero_grad(set_to_none=True)\n                output = model(x.to(device))\n                logits = output if branch == "effnet" else output["logits"]\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels.to(device, dtype=torch.float32))\n                if not torch.isfinite(loss):\n                    raise ValueError("Non-finite training loss")\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0, error_if_nonfinite=True)\n                optimizer.step()\n                loss_sum += float(loss.detach()) * len(labels)\n                count += len(labels)\n            predictions = predict_branch(model, val_loader, validation, branch, device)\n            auc = float(roc_auc_score(predictions.label, predictions[score_column]))\n            row = {"branch": branch, "stage": stage, "epoch": epoch, "train_loss": loss_sum / count,\n                   "validation_auc": auc, "seconds_including_validation": time.perf_counter() - epoch_start,\n                   "trainable_parameters": trainable}\n            history.append(row)\n            print(row)\n            pd.DataFrame(history).to_csv(Path(run) / f"{branch}_history.csv", index=False)\n            if auc > best_auc:\n                best_auc = auc\n                selected = {"stage": stage, "epoch": epoch, "validation_auc": auc}\n                torch.save({"model_state": model.state_dict(), **selected}, checkpoint)\n    if str(device).startswith("cuda"):\n        torch.cuda.synchronize()\n    elapsed = time.perf_counter() - start\n    peak = torch.cuda.max_memory_allocated() if str(device).startswith("cuda") else None\n    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True)["model_state"], strict=True)\n    model.cpu().eval()\n    report = {"branch": branch, "selected": selected, "training_images": len(branch_frame),\n              "adaptation_seconds_including_validation_and_saves": elapsed,\n              "max_trainable_parameters": max_trainable, "total_parameters": sum(p.numel() for p in model.parameters()),\n              "cuda_peak_allocated_bytes": peak, "checkpoint_sha256": sha256_file(checkpoint)}\n    write_json(Path(run) / f"{branch}_training.json", report)\n    return report\n\n\ndef fit_variant(name, models, processor, partition, validation, config, run, device):\n    import joblib\n    fusion_rows = partition[partition.role == "fusion_train"]\n    branch_ids = partition.loc[partition.role == "branch_train", "sample_id"]\n    train_predictions = predict_pair(models, processor, fusion_rows, config, device)\n    validation_predictions = predict_pair(models, processor, validation, config, device)\n    start = time.perf_counter()\n    fusion = fit_fusion(train_predictions, branch_ids)\n    fit_seconds = time.perf_counter() - start\n    validation_predictions["fusion_prob"] = fusion.predict_proba(validation_predictions[FEATURES])[:, 1]\n    thresholds = {key: max_f1_threshold(validation_predictions.label, validation_predictions[key])\n                  for key in ("effnet_prob", "semantic_vit_prob", "fusion_prob")}\n    write_json(Path(run) / f"{name}_selection.json", {\n        "thresholds": thresholds, "selection_split": "validation", "fusion_fit_seconds": fit_seconds,\n        "fusion_training_images": len(fusion_rows), "feature_order": FEATURES,\n    })\n    joblib.dump(fusion, Path(run) / f"{name}_fusion.joblib")\n    train_predictions.to_csv(Path(run) / f"{name}_fusion_training_predictions.csv", index=False)\n    validation_predictions.to_csv(Path(run) / f"{name}_validation_predictions.csv", index=False)\n    return fusion, thresholds\n\n\ndef scored_metrics(labels, scores, threshold):\n    if not np.isfinite(scores).all() or ((np.asarray(scores) < 0) | (np.asarray(scores) > 1)).any():\n        raise ValueError("Invalid detection scores")\n    result = binary_metrics(labels, scores, threshold)\n    result["fpr"] = result["fp"] / (result["fp"] + result["tn"])\n    return result\n\n\ndef evaluate_variant(name, models, processor, frames, fusion, thresholds, config, run, device):\n    rows = []\n    for split in ("test", "unseen_generator"):\n        predictions = predict_pair(models, processor, frames[split], config, device)\n        predictions["fusion_prob"] = fusion.predict_proba(predictions[FEATURES])[:, 1]\n        predictions.to_csv(Path(run) / f"{name}_{split}_predictions.csv", index=False)\n        for column in ("effnet_prob", "semantic_vit_prob", "fusion_prob"):\n            rows.append({"model": f"{name}/{column}", "split": split,\n                         **scored_metrics(predictions.label, predictions[column], thresholds[column])})\n    return rows\n\n\ndef historical_references(frames, config, audit, run):\n    rows, evidence = [], {}\n    threshold_path = config.cf_root / "results/validation_thresholds.json"\n    thresholds = json.loads(threshold_path.read_text())\n    evidence[str(threshold_path)] = sha256_file(threshold_path)\n    old_metrics = [r for r in audit["metrics"] if "Full fusion" in r["model"]]\n    for split in ("test", "unseen_generator"):\n        for key in ("pretrained", "finetuned"):\n            path = config.cf_root / f"predictions/{key}_{split}.csv"\n            pred = align_predictions(frames[split], checked_csv(path), ["prob_fake"])\n            threshold = thresholds["baseline" if key == "pretrained" else key]["max_f1"]\n            rows.append({"model": f"CF/{key}", "split": split,\n                         **scored_metrics(pred.label, pred.prob_fake, threshold)})\n            evidence[str(path)] = sha256_file(path)\n        matches = [r for r in old_metrics if r["split"] == split]\n        if len(matches) != 1:\n            raise ValueError("Missing/ambiguous original full fusion reference")\n        path = config.audit_root / f"aishield_{split}_predictions.csv"\n        pred = align_predictions(frames[split], checked_csv(path), ["aishield_full_prob"])\n        rows.append({"model": "original_full_17_feature/reference_only", "split": split,\n                     **scored_metrics(pred.label, pred.aishield_full_prob, matches[0]["threshold"])})\n        evidence[str(path)] = sha256_file(path)\n    write_json(Path(run) / "historical_reference_hashes.json", evidence)\n    return rows\n\n\ndef benchmark(predict_one, paths, device, repeats=3, warmup=3):\n    """Warm batch-1 end-to-end latency; excludes checkpoint/model loading."""\n    import torch\n    import psutil\n    if not paths or repeats < 1:\n        raise ValueError("Need benchmark images and positive repeats")\n    def sync():\n        if str(device).startswith("cuda"):\n            torch.cuda.synchronize()\n    for path in paths[:warmup]:\n        predict_one(path)\n    sync()\n    if str(device).startswith("cuda"):\n        torch.cuda.reset_peak_memory_stats()\n    durations = []\n    for _ in range(repeats):\n        for path in paths:\n            sync()\n            start = time.perf_counter()\n            predict_one(path)\n            sync()\n            durations.append(time.perf_counter() - start)\n    return {"images": len(paths), "repeats": repeats, "batch_size": 1,\n            "median_seconds": float(np.median(durations)), "p95_seconds": float(np.quantile(durations, .95)),\n            "cuda_peak_allocated_bytes": torch.cuda.max_memory_allocated() if str(device).startswith("cuda") else None,\n            "process_rss_after_bytes": psutil.Process().memory_info().rss,\n            "memory_note": "RSS is the whole notebook process after this run, NOT isolated model peak memory.",\n            "scope": "warm file-open + preprocessing + transfer + model + score; no model load, API, provenance or network"}\n\n\nclass PairPredictor:\n    def __init__(self, models, processor, fusion, device):\n        from torchvision import transforms\n        self.models, self.processor, self.fusion, self.device = models, processor, fusion, device\n        self.transform = transforms.Compose([\n            transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),\n            transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),\n        ])\n        for model in models.values():\n            model.to(device).eval()\n\n    def predict(self, path):\n        import torch\n        image = open_rgb(path)\n        with torch.inference_mode():\n            logit = self.models["effnet"](self.transform(image).unsqueeze(0).to(self.device))[0]\n            semantic = self.models["semantic"](self.processor(images=image, return_tensors="pt")["pixel_values"].to(self.device))\n            row = {"effnet_logit": float(logit), "effnet_prob": float(torch.sigmoid(logit)),\n                   "semantic_vit_logit": float(semantic["logits"][0]),\n                   "semantic_vit_prob": float(torch.sigmoid(semantic["logits"][0]))}\n            for key in ("attention_entropy", "attention_focus", "self_attention_entropy", "self_attention_focus"):\n                row["semantic_" + key] = float(semantic[key][0])\n        values = pd.DataFrame([row])[FEATURES]\n        if not np.isfinite(values.to_numpy()).all():\n            raise ValueError("Non-finite inference features")\n        return float(self.fusion.predict_proba(values)[0, 1])\n', 'src/ai_shield_adaptation_models.py': '"""Research models: original semantic head, audited loaders, strict checkpoints."""\nimport math\nfrom contextlib import nullcontext\nimport torch\nfrom torch import nn\n\nfrom src.ai_shield_adaptation import SEMANTIC_ID, SEMANTIC_REVISION\n\n\nclass CustomMultiHeadSelfAttention(nn.Module):\n    def __init__(self, d_model, num_heads, dropout=0.15):\n        super().__init__()\n        if d_model % num_heads != 0:\n            raise ValueError(f"d_model={d_model} must be divisible by num_heads={num_heads}")\n        self.d_model = d_model\n        self.num_heads = num_heads\n        self.head_dim = d_model // num_heads\n        self.scale = self.head_dim ** -0.5\n\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def _split_heads(self, x):\n        batch_size, num_tokens, _ = x.shape\n        x = x.view(batch_size, num_tokens, self.num_heads, self.head_dim)\n        return x.transpose(1, 2)\n\n    def _merge_heads(self, x):\n        batch_size, num_heads, num_tokens, head_dim = x.shape\n        x = x.transpose(1, 2).contiguous()\n        return x.view(batch_size, num_tokens, num_heads * head_dim)\n\n    def forward(self, x):\n        q = self._split_heads(self.q_proj(x))\n        k = self._split_heads(self.k_proj(x))\n        v = self._split_heads(self.v_proj(x))\n\n        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale\n        attention = torch.softmax(scores, dim=-1)\n        attention_for_summary = attention\n        attention = self.dropout(attention)\n\n        context = torch.matmul(attention, v)\n        context = self._merge_heads(context)\n        output = self.out_proj(context)\n        return output, attention_for_summary\n\n\nclass SemanticTransformerBlock(nn.Module):\n    def __init__(self, d_model, num_heads, mlp_ratio=2.0, dropout=0.15):\n        super().__init__()\n        hidden_dim = int(d_model * mlp_ratio)\n        self.norm1 = nn.LayerNorm(d_model)\n        self.attention = CustomMultiHeadSelfAttention(d_model, num_heads, dropout)\n        self.dropout1 = nn.Dropout(dropout)\n        self.norm2 = nn.LayerNorm(d_model)\n        self.mlp = nn.Sequential(\n            nn.Linear(d_model, hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(hidden_dim, d_model),\n            nn.Dropout(dropout),\n        )\n\n    def forward(self, x):\n        attn_input = self.norm1(x)\n        attn_output, attention = self.attention(attn_input)\n        x = x + self.dropout1(attn_output)\n        mlp_input = self.norm2(x)\n        x = x + self.mlp(mlp_input)\n        return x, attention\n\n\nclass SemanticTransformerHead(nn.Module):\n    def __init__(\n        self,\n        hidden_size,\n        d_model=256,\n        num_heads=4,\n        depth=1,\n        mlp_ratio=2.0,\n        attention_dim=256,\n        classifier_hidden_dim=256,\n        dropout=0.15,\n    ):\n        super().__init__()\n        self.input_projection = nn.Linear(hidden_size, d_model)\n        self.blocks = nn.ModuleList([\n            SemanticTransformerBlock(d_model, num_heads, mlp_ratio, dropout)\n            for _ in range(depth)\n        ])\n        self.final_norm = nn.LayerNorm(d_model)\n\n        self.pool_key = nn.Linear(d_model, attention_dim)\n        self.pool_energy = nn.Linear(attention_dim, 1)\n        self.classifier = nn.Sequential(\n            nn.Dropout(dropout),\n            nn.Linear(d_model, classifier_hidden_dim),\n            nn.GELU(),\n            nn.Dropout(dropout),\n            nn.Linear(classifier_hidden_dim, 1),\n        )\n\n    def _attention_summary(self, attention_maps):\n        eps = 1e-8\n        summaries = []\n        for attention in attention_maps:\n            num_tokens = attention.shape[-1]\n            entropy = -(attention * torch.log(attention + eps)).sum(dim=-1)\n            entropy = entropy / math.log(num_tokens)\n            focus = attention.max(dim=-1).values\n            summaries.append((entropy.mean(dim=(1, 2)), focus.mean(dim=(1, 2))))\n        entropy = torch.stack([item[0] for item in summaries], dim=0).mean(dim=0)\n        focus = torch.stack([item[1] for item in summaries], dim=0).mean(dim=0)\n        return entropy, focus\n\n    def forward(self, patch_tokens):\n        x = self.input_projection(patch_tokens)\n        self_attention_maps = []\n        for block in self.blocks:\n            x, attention = block(x)\n            self_attention_maps.append(attention)\n        x = self.final_norm(x)\n\n        pool_energy = self.pool_energy(torch.tanh(self.pool_key(x))).squeeze(-1)\n        pool_weights = torch.softmax(pool_energy, dim=1)\n        pooled = torch.sum(pool_weights.unsqueeze(-1) * x, dim=1)\n        logits = self.classifier(pooled).squeeze(-1)\n\n        eps = 1e-8\n        pool_entropy = -(pool_weights * torch.log(pool_weights + eps)).sum(dim=1)\n        pool_entropy = pool_entropy / math.log(pool_weights.shape[1])\n        pool_focus = pool_weights.max(dim=1).values\n        self_entropy, self_focus = self._attention_summary(self_attention_maps)\n\n        return {\n            "logits": logits,\n            "attention_weights": pool_weights,\n            "attention_entropy": pool_entropy,\n            "attention_focus": pool_focus,\n            "self_attention_entropy": self_entropy,\n            "self_attention_focus": self_focus,\n            "semantic_feature": pooled,\n        }\n\nclass EfficientNetBinaryClassifier(nn.Module):\n    def __init__(self):\n        super().__init__()\n        import timm\n        self.model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=1, drop_rate=0.2)\n\n    def forward(self, x):\n        return self.model(x).squeeze(1)\n\n\nclass ViTSemanticAttentionClassifier(nn.Module):\n    def __init__(self, backbone):\n        super().__init__()\n        self.vit = backbone\n        self.semantic_head = SemanticTransformerHead(hidden_size=backbone.config.hidden_size)\n        self.vit.requires_grad_(False)\n\n    def forward(self, pixel_values):\n        # The saved audit used cuDNN\'s default TF32 convolution arithmetic.\n        # Scope it to ViT so CF and the other branches retain their own settings.\n        precision = torch.backends.cudnn.flags(\n            enabled=torch.backends.cudnn.enabled,\n            benchmark=torch.backends.cudnn.benchmark,\n            deterministic=torch.backends.cudnn.deterministic,\n            allow_tf32=True,\n        ) if pixel_values.device.type == "cuda" else nullcontext()\n        with precision:\n            output = self.vit(pixel_values=pixel_values, return_dict=True)\n        return self.semantic_head(output.last_hidden_state[:, 1:, :])\n\n\ndef load_original_models(config):\n    from transformers import AutoImageProcessor, AutoModel\n    # Original files are user\'s own training artifacts; preflight verifies their hashes.\n    effnet = EfficientNetBinaryClassifier()\n    saved = torch.load(config.checkpoints["effnet"], map_location="cpu", weights_only=False)\n    state = saved["model_state"]\n    if not all(k.startswith("model.") for k in state):\n        state = {"model." + k: v for k, v in state.items()}\n    effnet.load_state_dict(state, strict=True)\n\n    processor = AutoImageProcessor.from_pretrained(SEMANTIC_ID, revision=SEMANTIC_REVISION, use_fast=True)\n    backbone = AutoModel.from_pretrained(SEMANTIC_ID, revision=SEMANTIC_REVISION)\n    semantic = ViTSemanticAttentionClassifier(backbone)\n    saved = torch.load(config.checkpoints["semantic"], map_location="cpu", weights_only=False)\n    if saved.get("config", {}).get("freeze_vit_backbone", True) is not True:\n        raise ValueError("Cannot restore only the head of a trained backbone")\n    state = {k.removeprefix("semantic_head."): v for k, v in saved["model_state"].items()\n             if k.startswith("semantic_head.")}\n    if not state:\n        raise ValueError("Semantic head weights are missing")\n    semantic.semantic_head.load_state_dict(state, strict=True)\n    return {"effnet": effnet.eval(), "semantic": semantic.eval()}, processor\n\n', 'src/ai_shield_metrics.py': '"""Shared metrics for AI Shield binary detectors and fusion models."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    brier_score_loss,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\n\n\ndef binary_metrics(y_true, y_prob, threshold: float = 0.5) -> dict[str, float | int]:\n    """Compute the binary metrics used across the AI Shield notebooks."""\n    y_true_array = np.asarray(y_true).astype(int)\n    y_prob_array = np.asarray(y_prob).astype(float)\n    y_pred = (y_prob_array >= threshold).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y_true_array, y_pred, labels=[0, 1]).ravel()\n    return {\n        "auc": float(roc_auc_score(y_true_array, y_prob_array)),\n        "average_precision": float(average_precision_score(y_true_array, y_prob_array)),\n        "accuracy": float(accuracy_score(y_true_array, y_pred)),\n        "precision": float(precision_score(y_true_array, y_pred, zero_division=0)),\n        "recall": float(recall_score(y_true_array, y_pred, zero_division=0)),\n        "f1": float(f1_score(y_true_array, y_pred, zero_division=0)),\n        "brier": float(brier_score_loss(y_true_array, y_prob_array)),\n        "threshold": float(threshold),\n        "n": int(len(y_true_array)),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef threshold_sweep(y_true, y_prob, thresholds=None, metric: str = "f1") -> tuple[float, list[dict[str, float | int]]]:\n    """Evaluate metrics across thresholds and return the best threshold."""\n    if thresholds is None:\n        thresholds = np.linspace(0.01, 0.99, 99)\n    rows = [binary_metrics(y_true, y_prob, float(threshold)) for threshold in thresholds]\n    best = max(rows, key=lambda row: float(row[metric]))\n    return float(best["threshold"]), rows\n', 'src/ai_shield_cf_inference.py': '"""Frozen Community Forensics inference, matching finetune_v1 evaluation."""\nimport hashlib\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nimport timm\nfrom PIL import Image, ImageOps\nfrom torchvision import transforms\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\nclass CommunityForensicsDetector(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.vit = timm.create_model(\n            "vit_small_patch16_384.augreg_in21k_ft_in1k", pretrained=False,\n        )\n        self.vit.head = nn.Linear(self.vit.num_features, 1)\n\n    def forward(self, images):\n        return self.vit(images)\n\n\nclass FrozenCF:\n    def __init__(self, checkpoint_path, expected_sha256, threshold,\n                 expected_manifest_sha256, device=None):\n        if file_sha256(checkpoint_path) != expected_sha256:\n            raise RuntimeError("Checkpoint SHA-256 mismatch.")\n        if not 0 <= threshold <= 1:\n            raise ValueError("Threshold must be in [0, 1].")\n        self.threshold = float(threshold)\n        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\n        if self.device.type == "cuda":\n            torch.backends.cuda.matmul.allow_tf32 = False\n            torch.backends.cudnn.allow_tf32 = False\n        self.model = CommunityForensicsDetector()\n        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        if payload.get("stage") != "stage2_last_blocks":\n            raise RuntimeError("Checkpoint is not Stage 2.")\n        if payload.get("experiment_manifest_sha256") != expected_manifest_sha256:\n            raise RuntimeError("Checkpoint belongs to a different experiment manifest.")\n        self.best_val_auc = float(payload["best_val_auc"])\n        self.model.load_state_dict(payload["model_state"], strict=True)\n        self.model.requires_grad_(False)\n        self.model.float().eval().to(self.device)\n        self.preprocess = transforms.Compose([\n            transforms.Resize(440, interpolation=transforms.InterpolationMode.BILINEAR,\n                              antialias=True),\n            transforms.CenterCrop(384),\n            transforms.ToTensor(),\n            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),\n        ])\n\n    def predict(self, path):\n        with Image.open(path) as source:\n            source.seek(0)\n            image = ImageOps.exif_transpose(source).convert("RGB")\n            tensor = self.preprocess(image).unsqueeze(0).to(self.device)\n        with torch.inference_mode():\n            logits = self.model(tensor).reshape(-1)\n            if logits.numel() != 1 or not torch.isfinite(logits).all():\n                raise RuntimeError("Invalid model output.")\n            score = float(torch.sigmoid(logits)[0].cpu())\n            logit = float(logits[0].cpu())\n        return {"logit": logit, "prob_fake": score,\n                "predicted_label": int(score >= self.threshold)}\n', 'src/ai_shield_full_adaptation.py': '"""Full 17-feature adaptation; original artifacts and app remain read-only."""\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport shutil\nimport time\nimport uuid\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom src.ai_shield_adaptation import (\n    Config, FEATURES, SPLIT_HASHES, align_predictions, checked_csv, max_f1_threshold,\n    open_rgb, partition_training, predict_pair, preflight, scored_metrics,\n    sha256_file, write_json,\n)\n\nAUX_FEATURES = [\'vae_recon_error\', \'vae_recon_percentile\', \'face_present\',\n    \'num_faces\', \'largest_face_ratio\', \'face_confidence\',\n    \'openforensics_applicable\', \'openforensics_prob_fake\']\nFULL_FEATURES = FEATURES[:2] + [\'effnet_calibrated_prob\'] + AUX_FEATURES + FEATURES[2:]\n\n\n@dataclass(frozen=True)\nclass FullConfig(Config):\n    # About 70% branch / 15% calibration / 15% fusion, grouped before fitting.\n    fusion_fraction: float = 0.30\n\n    @property\n    def extras(self):\n        phase1 = self.root / \'AI_Shield_Phase1\'\n        phase3 = self.root / \'AI_Shield_Phase3_Fusion/forensic_semantic_glm_v1\'\n        return {\n            \'vae\': phase1 / \'vae_real_only_tinygenimage_v1/vae_real_only_tinygenimage_v1_best.pt\',\n            \'vae_reference_errors\': phase1 / \'vae_real_only_tinygenimage_v1/vae_reconstruction_errors.npz\',\n            \'platt_calibrator\': phase1 / \'effnet_b0_tinygenimage_full_balanced_v1/platt_calibrator.pkl\',\n            \'openforensics_h5\': phase1 / \'openforensics_effnetv2b0/dffnetv2B0.h5\',\n            \'openforensics_json\': phase1 / \'openforensics_effnetv2b0/dffnetv2B0.json\',\n            \'full_glm\': phase3 / \'checkpoints/phase3_full_forensic_semantic_glm_v1.pkl\',\n            \'phase3_metrics\': phase3 / \'results/phase3_fusion_metrics_v1.json\',\n        }\n\n\ndef full_preflight(config):\n    frames, audit = preflight(config)\n    for key, path in config.extras.items():\n        if not path.is_file():\n            raise FileNotFoundError(f\'Missing original {key}: {path}\')\n        expected = audit[\'source_artifacts\'].get(key + \'_sha256\')\n        if expected and sha256_file(path) != expected:\n            raise ValueError(f\'Original {key} differs from the completed audit\')\n    for split in (\'test\', \'unseen_generator\'):\n        saved = checked_csv(config.audit_root / f\'aishield_{split}_predictions.csv\')\n        align_predictions(frames[split], saved, FULL_FEATURES + [\'aishield_full_prob\'])\n    return frames, audit\n\n\ndef full_partition(train, config):\n    partition = partition_training(train, config.fusion_fraction, config.seed)\n    reserve = partition[partition.role == \'fusion_train\'].copy()\n    divided = partition_training(reserve, 0.5, config.seed + 1)\n    roles = divided.set_index(\'sample_id\').role.replace({\'branch_train\': \'calibration\'})\n    selected = partition.sample_id.isin(roles.index)\n    partition.loc[selected, \'role\'] = partition.loc[selected, \'sample_id\'].map(roles)\n    if set(partition.role) != {\'branch_train\', \'calibration\', \'fusion_train\'}:\n        raise ValueError(\'Three independent training roles are required\')\n    return partition\n\n\ndef prepare_full_run(config, frames):\n    if config.head_epochs < 1 or config.partial_epochs < 0 or config.batch_size < 1:\n        raise ValueError(\'Invalid training budget\')\n    partition = full_partition(frames[\'train\'], config)\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\') + \'_\' + uuid.uuid4().hex[:8]\n    run = config.root / \'AI_Shield_Adaptation/full_pipeline_v1\' / stamp\n    run.mkdir(parents=True, exist_ok=False)\n    partition.to_csv(run / \'training_roles.csv\', index=False)\n    inputs = {**config.checkpoints, **config.extras}\n    for split in (\'test\', \'unseen_generator\'):\n        inputs[\'audit_\' + split] = config.audit_root / f\'aishield_{split}_predictions.csv\'\n    write_json(run / \'input_hashes.json\', {k: {\'path\': str(p), \'sha256\': sha256_file(p)} for k, p in inputs.items()})\n    write_json(run / \'experiment_plan.json\', {\n        \'config\': asdict(config), \'feature_order\': FULL_FEATURES, \'split_sha256\': SPLIT_HASHES,\n        \'role_counts\': partition.groupby([\'role\', \'label\']).size().rename(\'n\').reset_index().to_dict(\'records\'),\n        \'libraries\': {n: importlib.metadata.version(n) for n in (\'torch\', \'torchvision\', \'timm\', \'transformers\', \'scikit-learn\', \'numpy\', \'pandas\', \'Pillow\')},\n        \'scope\': \'Original full 17-feature architecture; adapt EfficientNet and semantic head/backbone, refit calibration and full fusion; VAE and face model frozen.\',\n        \'budget\': \'Same overall 2990-image training pool as CF, fewer branch-training images; compute and pretraining are not matched.\',\n        \'vae_protocol\': \'Preserve sampled-latent VAE with one reproducible CPU sample seeded by image hash on every split and in live inference. Original historical results retain their original cached draws as a separate reference.\',\n        \'test_policy\': \'Previously inspected historical tests; no training, calibration, threshold or checkpoint selection on them.\',\n    })\n    return run, partition\n\n\ndef verify_input_lock(config, run):\n    for key, item in json.loads((Path(run) / \'input_hashes.json\').read_text()).items():\n        if sha256_file(item[\'path\']) != item[\'sha256\']:\n            raise ValueError(f\'Original input changed during the run: {key}\')\n\n\ndef stage_images(frames, destination):\n    """Copy verified bytes to runtime disk, avoiding repeated Drive I/O in epochs."""\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=False)\n    total = sum(Path(r.local_path).stat().st_size for f in frames.values() for r in f.itertuples())\n    if shutil.disk_usage(destination).free < total + 1024 ** 3:\n        raise RuntimeError(\'Not enough runtime disk; set STAGE_TO_LOCAL=False in setup\')\n    staged = {}\n    from tqdm.auto import tqdm\n    for split, frame in frames.items():\n        copy = frame.copy()\n        paths = []\n        for row in tqdm(frame.itertuples(), total=len(frame), desc=\'Stage \' + split):\n            target = destination / (row.sample_id + Path(row.local_path).suffix)\n            shutil.copyfile(row.local_path, target)\n            if sha256_file(target) != row.sha256:\n                raise ValueError(\'Copied bytes changed: \' + row.sample_id)\n            paths.append(str(target))\n        copy[\'local_path\'] = paths\n        staged[split] = copy\n    return staged\n\n\ndef load_legacy_fusion(path):\n    # This is an owned artifact whose hash is checked against the previous audit.\n    model = joblib.load(path)\n    for _, step in getattr(model, \'steps\', []):\n        if step.__class__.__name__ == \'SimpleImputer\' and not hasattr(step, \'_fill_dtype\'):\n            step._fill_dtype = np.asarray(step.statistics_).dtype\n    if list(model.feature_names_in_) != FULL_FEATURES:\n        raise ValueError(\'Saved fusion feature order differs from the 17-feature schema\')\n    return model\n\n\ndef replay_original(config, frames, run):\n    model = load_legacy_fusion(config.extras[\'full_glm\'])\n    checks = []\n    for split in (\'test\', \'unseen_generator\'):\n        saved = align_predictions(frames[split], checked_csv(config.audit_root / f\'aishield_{split}_predictions.csv\'), FULL_FEATURES + [\'aishield_full_prob\'])\n        actual = model.predict_proba(saved[FULL_FEATURES])[:, 1]\n        if not np.allclose(actual, saved.aishield_full_prob, atol=1e-7, rtol=1e-6):\n            raise ValueError(\'Original full fusion does not reproduce saved scores: \' + split)\n        checks.append({\'split\': split, \'n\': len(saved), \'max_absolute_error\': float(np.max(np.abs(actual - saved.aishield_full_prob)))})\n    write_json(Path(run) / \'original_full_fusion_replay.json\', checks)\n    return checks\n\n\ndef latent_seed(identity):\n    return int(hashlib.sha256((\'vae-4242:\' + str(identity)).encode()).hexdigest()[:8], 16)\n\n\nclass FrozenAuxiliary:\n    """Original sampled VAE + Haar face gate + original OpenForensics, all frozen."""\n    def __init__(self, config):\n        import torch\n        from torch import nn\n        from torchvision import transforms\n        import cv2\n\n        class ConvVAE(nn.Module):\n            def __init__(self):\n                super().__init__()\n                self.encoder = nn.Sequential(nn.Conv2d(3,32,4,2,1),nn.ReLU(True),nn.Conv2d(32,64,4,2,1),nn.ReLU(True),nn.Conv2d(64,128,4,2,1),nn.ReLU(True),nn.Conv2d(128,256,4,2,1),nn.ReLU(True))\n                self.fc_mu = nn.Linear(16384,128)\n                self.fc_logvar = nn.Linear(16384,128)\n                self.fc_decode = nn.Linear(128,16384)\n                self.decoder = nn.Sequential(nn.ConvTranspose2d(256,128,4,2,1),nn.ReLU(True),nn.ConvTranspose2d(128,64,4,2,1),nn.ReLU(True),nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(True),nn.ConvTranspose2d(32,3,4,2,1),nn.Sigmoid())\n            def encode(self, x):\n                h = self.encoder(x).view(x.shape[0], -1)\n                return self.fc_mu(h), self.fc_logvar(h)\n            def decode(self, z):\n                return self.decoder(self.fc_decode(z).view(z.shape[0],256,8,8))\n\n        self.vae = ConvVAE()\n        self.vae.load_state_dict(torch.load(config.extras[\'vae\'], map_location=\'cpu\', weights_only=False)[\'model_state\'], strict=True)\n        self.vae.eval().requires_grad_(False)\n        self.vae_transform = transforms.Compose([transforms.Resize((128,128)), transforms.ToTensor()])\n        with np.load(config.extras[\'vae_reference_errors\'], allow_pickle=False) as saved:\n            self.reference = np.sort(np.asarray(saved[\'full\'], dtype=float).reshape(-1))\n        if not len(self.reference) or not np.isfinite(self.reference).all():\n            raise ValueError(\'Invalid original VAE reference distribution\')\n        cascade = Path(cv2.data.haarcascades) / \'haarcascade_frontalface_default.xml\'\n        self.face = cv2.CascadeClassifier(str(cascade))\n        if self.face.empty():\n            raise RuntimeError(\'Cannot load the original Haar face gate\')\n        os.environ[\'TF_CPP_MIN_LOG_LEVEL\'] = \'2\'\n        import tensorflow as tf\n        tf.config.set_visible_devices([], \'GPU\')\n        self.tf = tf\n        with tf.device(\'/CPU:0\'):\n            try:\n                self.face_model = tf.keras.models.model_from_json(config.extras[\'openforensics_json\'].read_text())\n                self.face_model.load_weights(str(config.extras[\'openforensics_h5\']))\n            except (ValueError, TypeError, KeyError):\n                # Same legacy reconstruction used successfully in the saved audit.\n                self.face_model = tf.keras.Sequential([\n                    tf.keras.applications.EfficientNetV2B0(weights=None, input_shape=(256,256,3), include_top=False, pooling=\'max\', include_preprocessing=True),\n                    tf.keras.layers.Dense(1, activation=\'sigmoid\', name=\'dense_2\')], name=\'dffnetv2b0\')\n                self.face_model(tf.zeros((1,256,256,3)), training=False)\n                self.face_model.load_weights(str(config.extras[\'openforensics_h5\']))\n        self.face_model.trainable = False\n\n    def face_features(self, image):\n        import cv2\n        gray = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY)\n        try:\n            rects, _, weights = self.face.detectMultiScale3(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40,40), outputRejectLevels=True)\n            weights = list(weights) if len(rects) else []\n        except cv2.error:\n            rects = self.face.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40,40))\n            weights = [1.0] * len(rects)\n        valid = [(float(w*h/(image.width*image.height)), float(c)) for (_,_,w,h),c in zip(rects,weights) if c >= 0 and w*h/(image.width*image.height) >= .01]\n        score = .5\n        if valid:\n            batch = np.asarray(image.resize((256,256)), dtype=np.float32)[None]\n            with self.tf.device(\'/CPU:0\'):\n                p = np.asarray(self.face_model(batch, training=False)).reshape(-1)\n            if len(p) != 1 or not np.isfinite(p).all() or not 0 <= p[0] <= 1:\n                raise ValueError(\'Invalid OpenForensics output\')\n            score = 1.0 - float(p[0])\n        return {\'face_present\': int(bool(valid)), \'num_faces\': len(valid),\n            \'largest_face_ratio\': max([v[0] for v in valid], default=0.),\n            \'face_confidence\': max([v[1] for v in valid], default=0.),\n            \'openforensics_applicable\': int(bool(valid)), \'openforensics_prob_fake\': score}\n\n    def features(self, image, identity):\n        import torch\n        with torch.inference_mode():\n            x = self.vae_transform(image)[None]\n            mu, logvar = self.vae.encode(x)\n            rng = torch.Generator(device=\'cpu\').manual_seed(latent_seed(identity))\n            eps = torch.randn(mu.shape, generator=rng, dtype=mu.dtype)\n            reconstruction = self.vae.decode(mu + eps * torch.exp(.5 * logvar))\n            error = float((x - reconstruction).square().mean())\n        if not np.isfinite(error):\n            raise ValueError(\'Non-finite VAE reconstruction\')\n        return {\'vae_recon_error\': error, \'vae_recon_percentile\': float(np.searchsorted(self.reference,error,side=\'right\')/len(self.reference)), **self.face_features(image)}\n\n\ndef verify_face_parity(aux, config, frames, run):\n    saved = checked_csv(config.audit_root / \'aishield_test_predictions.csv\')\n    selected = pd.concat([saved[saved.face_present == 1].head(3), saved[saved.face_present == 0].head(3)])\n    if set(selected.face_present) != {0,1}:\n        raise ValueError(\'Need both face and no-face reference cases\')\n    paths = frames[\'test\'].set_index(\'sample_id\').local_path\n    errors = []\n    for r in selected.itertuples():\n        actual = aux.face_features(open_rgb(paths.loc[r.sample_id]))\n        for key, value in actual.items():\n            expected = getattr(r, key)\n            if not np.isclose(value, expected, atol=5e-4, rtol=1e-4):\n                raise ValueError(f\'Face parity failed: {r.sample_id} / {key}\')\n        errors.append({\'sample_id\': r.sample_id, \'status\': \'pass\'})\n    write_json(Path(run) / \'face_parity.json\', errors)\n    return errors\n\n\ndef cache_auxiliary(aux, frames, config, run):\n    from tqdm.auto import tqdm\n    tables = {}\n    for split, frame in frames.items():\n        rows = []\n        for r in tqdm(frame.itertuples(), total=len(frame), desc=\'Frozen auxiliary: \'+split):\n            rows.append({\'sample_id\':r.sample_id,\'label\':r.label, **aux.features(open_rgb(r.local_path), r.sha256)})\n        result = align_predictions(frame, pd.DataFrame(rows), AUX_FEATURES)\n        result[\'split\'] = split\n        result.to_csv(Path(run) / f\'auxiliary_{split}.csv\', index=False)\n        tables[split] = result\n    write_json(Path(run) / \'auxiliary_cache_hashes.json\', {s:sha256_file(Path(run)/f\'auxiliary_{s}.csv\') for s in tables})\n    return tables\n\n\ndef combine_features(main, aux, calibrator):\n    chosen = aux[aux.sample_id.isin(main.sample_id)]\n    ordered = align_predictions(main, chosen, AUX_FEATURES)\n    result = main.copy()\n    for key in AUX_FEATURES:\n        result[key] = ordered[key].to_numpy()\n    result[\'effnet_calibrated_prob\'] = calibrator.predict_proba(result[[\'effnet_logit\']])[:,1]\n    align_predictions(result, result, FULL_FEATURES)\n    return result\n\n\ndef fit_full_models(calibration, fusion_main, aux, branch_ids):\n    if set(calibration.split) != {\'train\'} or set(fusion_main.split) != {\'train\'}:\n        raise ValueError(\'Calibration and fusion fitting must use training data only\')\n    sets = [set(calibration.sample_id),set(fusion_main.sample_id),set(branch_ids)]\n    if any(sets[a] & sets[b] for a,b in [(0,1),(0,2),(1,2)]):\n        raise ValueError(\'Branch, calibration and fusion images must be disjoint\')\n    for frame in (calibration, fusion_main):\n        align_predictions(frame, frame, FEATURES)\n        if set(frame.label) != {0,1}:\n            raise ValueError(\'Each fitting role needs both classes\')\n    calibrator = LogisticRegression(C=1.,max_iter=2000,random_state=191)\n    calibrator.fit(calibration[[\'effnet_logit\']],calibration.label)\n    full = combine_features(fusion_main, aux, calibrator)\n    fusion = make_pipeline(SimpleImputer(strategy=\'median\'), StandardScaler(), LogisticRegression(C=1.,max_iter=5000,random_state=191))\n    fusion.fit(full[FULL_FEATURES],full.label)\n    return calibrator, fusion, full\n\n\ndef fit_full_variant(name, models, processor, partition, frames, auxiliary, config, run, device):\n    development = partition[partition.role != \'branch_train\']\n    main = predict_pair(models,processor,development,config,device)\n    roles = partition.set_index(\'sample_id\').role\n    calibration = main[main.sample_id.map(roles) == \'calibration\']\n    fusion_main = main[main.sample_id.map(roles) == \'fusion_train\']\n    start = time.perf_counter()\n    calibrator, fusion, full_train = fit_full_models(calibration,fusion_main,auxiliary[\'train\'],partition.loc[partition.role==\'branch_train\',\'sample_id\'])\n    fit_seconds = time.perf_counter()-start\n    val = combine_features(predict_pair(models,processor,frames[\'validation\'],config,device),auxiliary[\'validation\'],calibrator)\n    val[\'fusion_prob\'] = fusion.predict_proba(val[FULL_FEATURES])[:,1]\n    thresholds = {k:max_f1_threshold(val.label,val[k]) for k in [\'effnet_prob\',\'effnet_calibrated_prob\',\'semantic_vit_prob\',\'fusion_prob\']}\n    joblib.dump(calibrator,Path(run)/f\'{name}_calibrator.joblib\')\n    joblib.dump(fusion,Path(run)/f\'{name}_fusion.joblib\')\n    calibration.to_csv(Path(run)/f\'{name}_calibration_predictions.csv\',index=False)\n    full_train.to_csv(Path(run)/f\'{name}_fusion_training_features.csv\',index=False)\n    val.to_csv(Path(run)/f\'{name}_validation_predictions.csv\',index=False)\n    selection = {\'thresholds\':thresholds,\'feature_order\':FULL_FEATURES,\'selection_split\':\'validation\',\n        \'fusion_training_n\':len(full_train),\'calibration_n\':len(calibration),\'fit_seconds\':fit_seconds,\n        \'validation_metrics\':scored_metrics(val.label,val.fusion_prob,thresholds[\'fusion_prob\'])}\n    write_json(Path(run)/f\'{name}_selection.json\',selection)\n    return calibrator,fusion,thresholds,selection\n\n\ndef evaluate_full_variant(name, models, processor, frames, auxiliary, calibrator, fusion, thresholds, config, run, device):\n    metrics, subgroups = [], []\n    for split in (\'test\',\'unseen_generator\'):\n        pred = combine_features(predict_pair(models,processor,frames[split],config,device),auxiliary[split],calibrator)\n        pred[\'fusion_prob\'] = fusion.predict_proba(pred[FULL_FEATURES])[:,1]\n        pred[\'generator\'] = frames[split].generator.to_numpy()\n        pred.to_csv(Path(run)/f\'{name}_{split}_predictions.csv\',index=False)\n        for key,threshold in thresholds.items():\n            metrics.append({\'model\':name+\'/\'+key,\'split\':split,**scored_metrics(pred.label,pred[key],threshold)})\n        for generator,g in pred.groupby(\'generator\'):\n            positives = (g.fusion_prob >= thresholds[\'fusion_prob\']).sum()\n            subgroups.append({\'model\':name,\'split\':split,\'generator\':generator,\'n\':len(g),\n                \'metric\':\'generated_recall\' if g.label.eq(1).all() else \'real_false_positive_rate\',\n                \'value\':float(positives/len(g))})\n    pd.DataFrame(subgroups).to_csv(Path(run)/f\'{name}_generator_breakdown.csv\',index=False)\n    return metrics,subgroups\n\n\ndef lock_full_selection(run, selections):\n    run = Path(run)\n    names = [\'effnet_adapted.pt\',\'semantic_adapted.pt\',\'input_hashes.json\',\'training_roles.csv\',\'auxiliary_cache_hashes.json\']\n    names += [f\'auxiliary_{s}.csv\' for s in (\'train\',\'validation\',\'test\',\'unseen_generator\')]\n    for variant in selections:\n        names += [f\'{variant}_{suffix}\' for suffix in (\'calibrator.joblib\',\'fusion.joblib\',\'selection.json\')]\n    winner = max(sorted(selections),key=lambda v:(selections[v][\'validation_metrics\'][\'f1\'],selections[v][\'validation_metrics\'][\'auc\']))\n    lock = {\'files\':{n:sha256_file(run/n) for n in names},\'validation_selected_candidate\':winner,\n        \'rule\':\'Highest validation F1, then validation AUROC; never select using test results.\'}\n    write_json(run/\'selection_lock.json\',lock)\n    return lock\n\n\ndef verify_full_lock(run):\n    lock = json.loads((Path(run)/\'selection_lock.json\').read_text())\n    for name,digest in lock[\'files\'].items():\n        if sha256_file(Path(run)/name) != digest:\n            raise ValueError(\'Locked experiment file changed: \'+name)\n    return lock\n\n\nclass FullPredictor:\n    def __init__(self,models,processor,auxiliary,calibrator,fusion,device,semantic_batch_size=16):\n        from src.ai_shield_adaptation import PairPredictor\n        if isinstance(semantic_batch_size, bool) or not isinstance(semantic_batch_size, int) or semantic_batch_size < 1:\n            raise ValueError(\'semantic_batch_size must be a positive integer\')\n        self.semantic_batch_size = semantic_batch_size\n        self.pair = PairPredictor(models,processor,fusion,device)\n        self.auxiliary,self.calibrator,self.fusion = auxiliary,calibrator,fusion\n\n    def predict(self,path,identity=None):\n        import torch\n        pair = self.pair\n        image = open_rgb(path)\n        with torch.inference_mode():\n            logit = pair.models[\'effnet\'](pair.transform(image)[None].to(pair.device))[0]\n            # Match evaluation\'s physical batch size; keep only the first result.\n            # cuDNN/ViT numerical paths can differ for a physical batch of one.\n            pixels = pair.processor(images=image,return_tensors=\'pt\')[\'pixel_values\']\n            if pixels.shape[0] != 1:\n                raise ValueError(\'Standalone inference expects one image\')\n            pixels = pixels.repeat(self.semantic_batch_size, 1, 1, 1).to(pair.device)\n            sem = pair.models[\'semantic\'](pixels)\n            row = {\'effnet_logit\':float(logit),\'effnet_prob\':float(torch.sigmoid(logit)),\n                \'semantic_vit_logit\':float(sem[\'logits\'][0]),\'semantic_vit_prob\':float(torch.sigmoid(sem[\'logits\'][0]))}\n            for key in (\'attention_entropy\',\'attention_focus\',\'self_attention_entropy\',\'self_attention_focus\'):\n                row[\'semantic_\'+key] = float(sem[key][0])\n        row.update(self.auxiliary.features(image,identity or sha256_file(path)))\n        row[\'effnet_calibrated_prob\'] = float(self.calibrator.predict_proba(pd.DataFrame([row])[[\'effnet_logit\']])[0,1])\n        values = pd.DataFrame([row])[FULL_FEATURES]\n        if not np.isfinite(values.to_numpy()).all():\n            raise ValueError(\'Non-finite full-pipeline features\')\n        return float(self.fusion.predict_proba(values)[0,1])\n\n\ndef load_full_predictor(config,run,device=\'cpu\',variant=\'adapted_full\'):\n    import torch\n    from src.ai_shield_adaptation_models import load_original_models\n    verify_full_lock(run)\n    verify_input_lock(config,run)\n    if variant not in (\'adapted_full\',\'frozen_full\'):\n        raise ValueError(\'Unknown full-pipeline variant\')\n    models,processor = load_original_models(config)\n    if variant == \'adapted_full\':\n        for branch,model in models.items():\n            model.load_state_dict(torch.load(Path(run)/f\'{branch}_adapted.pt\',map_location=\'cpu\',weights_only=True)[\'model_state\'],strict=True)\n    saved_batch_size = int(json.loads((Path(run)/\'experiment_plan.json\').read_text())[\'config\'][\'batch_size\'])\n    return FullPredictor(models,processor,FrozenAuxiliary(config),joblib.load(Path(run)/f\'{variant}_calibrator.joblib\'),joblib.load(Path(run)/f\'{variant}_fusion.joblib\'),device,semantic_batch_size=saved_batch_size)\n', 'scripts/AI_Shield_Setup_Resume_v1.py': '"""Save/recover the existing full-pipeline setup without creating another run.\n\nExecute after section 3 once. In a new session, mount Drive and execute again\nafter installing the notebook dependencies. This restores setup, not optimizer\nstate. Every staged image is still checked against its recorded SHA-256.\n"""\nimport hashlib\nimport importlib\nimport json\nimport re\nimport sys\nimport uuid\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nSOURCE_FILES = (\n    \'src/__init__.py\', \'src/ai_shield_adaptation.py\',\n    \'src/ai_shield_adaptation_models.py\', \'src/ai_shield_metrics.py\',\n    \'src/ai_shield_cf_inference.py\', \'src/ai_shield_full_adaptation.py\',\n)\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(path, payload):\n    path = Path(path)\n    temporary = path.with_name(path.name + \'.tmp-\' + uuid.uuid4().hex)\n    temporary.write_text(json.dumps(payload, indent=2), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef save_setup_state(namespace):\n    required = (\'CONFIG\', \'RUN\', \'frames\', \'partition\')\n    missing = [name for name in required if name not in namespace]\n    if missing:\n        raise RuntimeError(\'Finish section 3 once before saving setup. Missing: \' + \', \'.join(missing))\n    run = Path(namespace[\'RUN\']).resolve()\n    root = Path(namespace[\'CONFIG\'].drive_root).resolve()\n    parent = root / \'AI_Shield_Adaptation/full_pipeline_v1\'\n    if run.parent != parent or not (run / \'experiment_plan.json\').is_file():\n        raise RuntimeError(\'RUN is not an existing full-pipeline experiment.\')\n    if json.loads((run / \'experiment_plan.json\').read_text())[\'config\'] != asdict(namespace[\'CONFIG\']):\n        raise RuntimeError(\'Current CONFIG differs from the saved experiment.\')\n    # These immutable small records identify the original experiment. No image\n    # is reread here; preparation already checked it before creating this RUN.\n    hashes = {name: digest(run / name) for name in (\n        \'training_roles.csv\', \'input_hashes.json\', \'experiment_plan.json\')}\n    sources = {name: digest(run / \'code\' / name) for name in SOURCE_FILES}\n    saved = {\'version\': 1, \'run_name\': run.name, \'config\': asdict(namespace[\'CONFIG\']),\n             \'record_hashes\': hashes, \'source_hashes\': sources}\n    atomic_json(run / \'setup_resume.json\', saved)\n    atomic_json(parent / \'active_setup.json\', {\'version\': 1, \'run_name\': run.name})\n    print(\'SETUP SAVED:\', run)\n    if namespace.get(\'_AI_SHIELD_INTEGRATED\'):\n        print(\'Section 3 will reuse this saved experiment on the next setup recovery.\')\n    else:\n        print(\'After a new runtime: mount Drive, restore dependencies if needed, and run this same cell instead of section 3.\')\n    return saved\n\n\ndef source_is_registered_patch(run, relative, before_hash, after_hash):\n    if relative != \'src/ai_shield_adaptation_models.py\':\n        return False\n    expected_path = (run / \'code\' / relative).resolve()\n    for record in run.glob(\'semantic_compatibility_fix_*.json\'):\n        try:\n            payload = json.loads(record.read_text())\n        except (ValueError, OSError):\n            continue\n        if payload.get(\'fix\') != \'fast_processor_and_scoped_cudnn_tf32\':\n            continue\n        for item in payload.get(\'files\', []):\n            if (Path(item[\'path\']).resolve() == expected_path and\n                    item[\'before_sha256\'] == before_hash and item[\'after_sha256\'] == after_hash):\n                return True\n    return False\n\n\ndef load_setup_record(root, run_name=None):\n    parent = Path(root).resolve() / \'AI_Shield_Adaptation/full_pipeline_v1\'\n    if run_name is None:\n        pointer = parent / \'active_setup.json\'\n        if not pointer.is_file():\n            raise RuntimeError(\'No setup has been saved yet. Run section 3 once, then this cell.\')\n        run_name = json.loads(pointer.read_text())[\'run_name\']\n    if not re.fullmatch(r\'\\d{8}T\\d{6}Z_[a-f0-9]{8}\', run_name):\n        raise ValueError(\'Invalid saved run name\')\n    run = (parent / run_name).resolve()\n    if run.parent != parent:\n        raise ValueError(\'Run path leaves the experiment folder\')\n    saved = json.loads((run / \'setup_resume.json\').read_text())\n    if saved.get(\'version\') != 1 or saved[\'run_name\'] != run_name:\n        raise ValueError(\'Unexpected setup record\')\n    if Path(saved[\'config\'][\'drive_root\']).resolve() != Path(root).resolve():\n        raise ValueError(\'This setup belongs to a different Drive root\')\n    for name in (\'training_roles.csv\', \'input_hashes.json\', \'experiment_plan.json\'):\n        if digest(run / name) != saved[\'record_hashes\'][name]:\n            raise ValueError(\'Saved experiment record changed: \' + name)\n    for relative in SOURCE_FILES:\n        actual = digest(run / \'code\' / relative)\n        expected = saved[\'source_hashes\'][relative]\n        if actual != expected and not source_is_registered_patch(run, relative, expected, actual):\n            raise ValueError(\'Saved source changed without a registered fix: \' + relative)\n    return run, saved\n\n\ndef stage_verified_images(frames, destination):\n    """Reuse verified local files; copy/hash missing files in the same read.\n\n    A crash leaves at most an uncommitted partial file. Completed image files\n    survive kernel-only restarts and can be reused without accessing Drive.\n    """\n    from tqdm.auto import tqdm\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=True)\n    stats = {\'reused_local\': 0, \'copied_from_drive\': 0}\n    staged = {}\n    for split, frame in frames.items():\n        paths = []\n        for row in tqdm(frame.itertuples(), total=len(frame), desc=\'Restore \' + split):\n            if not re.fullmatch(r\'[a-f0-9]{24}\', str(row.sample_id)):\n                raise ValueError(\'Unexpected image ID\')\n            source = Path(row.local_path)\n            target = destination / (str(row.sample_id) + source.suffix)\n            if target.resolve().parent != destination:\n                raise ValueError(\'Image path leaves the cache\')\n            if target.is_file() and digest(target) == row.sha256:\n                stats[\'reused_local\'] += 1\n            else:\n                temporary = target.with_name(target.name + \'.partial\')\n                h = hashlib.sha256()\n                # Only the helper\'s per-image temporary file is overwritten.\n                with source.open(\'rb\') as incoming, temporary.open(\'wb\') as outgoing:\n                    for chunk in iter(lambda: incoming.read(1024 * 1024), b\'\'):\n                        h.update(chunk)\n                        outgoing.write(chunk)\n                if h.hexdigest() != row.sha256:\n                    raise ValueError(\'Image bytes differ from the recorded dataset: \' + str(row.sample_id))\n                temporary.replace(target)\n                stats[\'copied_from_drive\'] += 1\n            paths.append(str(target))\n        staged[split] = frame.copy()\n        staged[split][\'local_path\'] = paths\n    return staged, stats\n\n\ndef restore_setup_state(namespace):\n    root = Path(namespace.get(\'AI_SHIELD_DRIVE_ROOT\', \'/content/drive/MyDrive\'))\n    if not root.is_dir():\n        raise RuntimeError("Mount Drive first: from google.colab import drive; drive.mount(\'/content/drive\')")\n    run, saved = load_setup_record(root, namespace.get(\'RESUME_RUN_NAME\'))\n    # Restoring arbitrary intermediate optimizer state is outside this helper.\n    if any((run / name).exists() for name in (\n        \'effnet_adapted.pt\', \'semantic_adapted.pt\', \'selection_lock.json\',\n        \'frozen_full_fusion.joblib\', \'adapted_full_fusion.joblib\')):\n        raise RuntimeError(\'This run has progressed beyond setup. Its results are preserved; this helper cannot resume training/optimizer state.\')\n    if namespace.get(\'CONFIG\') is not None and asdict(namespace[\'CONFIG\']) != saved[\'config\']:\n        raise RuntimeError(\'Current CONFIG differs from the saved setup. Do not mix experiments.\')\n    try:\n        import numpy as np\n        import pandas as pd\n        import torch\n        import timm, transformers, sklearn\n    except ImportError as error:\n        raise RuntimeError(\'Restore section 1 dependencies in this runtime, then rerun this cell.\') from error\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Reconnect to a GPU runtime before restoring this experiment.\')\n    # Load the exact saved implementation, including a registered semantic fix.\n    code_root = run / \'code\'\n    for key in list(sys.modules):\n        if key == \'src\' or key.startswith(\'src.\'):\n            del sys.modules[key]\n    sys.path.insert(0, str(code_root))\n    importlib.invalidate_caches()\n    adaptation = importlib.import_module(\'src.ai_shield_adaptation\')\n    full = importlib.import_module(\'src.ai_shield_full_adaptation\')\n    models_module = importlib.import_module(\'src.ai_shield_adaptation_models\')\n    config = full.FullConfig(**saved[\'config\'])\n    frames = {}\n    for split, expected in adaptation.SPLIT_HASHES.items():\n        path = config.split_root / (split + \'.csv\')\n        if digest(path) != expected:\n            raise ValueError(\'Original dataset manifest changed: \' + split)\n        frames[split] = adaptation.checked_csv(path)\n    adaptation.validate_frames(frames, adaptation.EXPECTED_COUNTS)\n    print(\'Restoring existing experiment:\', run.name, flush=True)\n    print(\'Checking original model files...\', flush=True)\n    full.verify_input_lock(config, run)\n    partition = adaptation.checked_csv(run / \'training_roles.csv\')\n    expected_roles = full.full_partition(frames[\'train\'], config)\n    columns = [\'sample_id\', \'label\', \'role\']\n    pd.testing.assert_frame_equal(partition[columns].reset_index(drop=True),\n                                  expected_roles[columns].reset_index(drop=True))\n    frames, stats = stage_verified_images(frames, Path(\'/content\') / (\'ai_shield_images_\' + run.name))\n    partition[\'local_path\'] = partition.sample_id.map(frames[\'train\'].set_index(\'sample_id\').local_path)\n    for module in (adaptation, full):\n        namespace.update({k: v for k, v in vars(module).items() if not k.startswith(\'_\')})\n    from IPython.display import display\n    audit = json.loads((config.audit_root / \'saved_pipeline_evaluation_report.json\').read_text(encoding=\'utf-8\'))\n    namespace.update(CONFIG=config, RUN=run, frames=frames, partition=partition, audit=audit,\n                     DEVICE=\'cuda\', STAGE_TO_LOCAL=True, CODE_ROOT=code_root,\n                     load_original_models=models_module.load_original_models,\n                     np=np, pd=pd, torch=torch, display=display,\n                     cache_auxiliary=reuse_complete_auxiliary)\n    adaptation.seed_everything(config.seed)\n    print(\'SETUP RESTORED:\', stats)\n    if namespace.get(\'_AI_SHIELD_INTEGRATED\'):\n        print(\'Continue with section 4. The compatibility check is included there.\')\n    else:\n        print(\'Next: run the semantic repair if still needed, then restore_reference_checks(globals()).\')\n        print(\'Do not rerun section 3: your existing RUN and training roles are restored.\')\n    return stats\n\n\ndef restore_reference_checks(namespace):\n    """Repeat small live GPU checks and keep all earlier reports intact."""\n    from src.ai_shield_adaptation import check_audit_parity, write_json\n    from src.ai_shield_adaptation_models import load_original_models\n    from src.ai_shield_full_adaptation import FrozenAuxiliary, replay_original, verify_face_parity\n    config, frames, run, device = (namespace[k] for k in (\'CONFIG\', \'frames\', \'RUN\', \'DEVICE\'))\n    reports = Path(run) / \'resume_checks\' / uuid.uuid4().hex[:12]\n    models, processor = load_original_models(config)\n    parity = check_audit_parity(models, processor, frames, config, device)\n    reports.mkdir(parents=True, exist_ok=False)\n    write_json(reports / \'main_branch_parity.json\', parity)\n    replay = replay_original(config, frames, reports)\n    auxiliary_model = FrozenAuxiliary(config)\n    face_parity = verify_face_parity(auxiliary_model, config, frames, reports)\n    namespace.update(models=models, processor=processor, parity=parity, replay=replay,\n                     auxiliary_model=auxiliary_model, face_parity=face_parity)\n    print(\'Reference checks passed. Continue with the auxiliary-cache cell in section 4.\')\n\n\ndef reuse_complete_auxiliary(aux, frames, config, run):\n    from src.ai_shield_adaptation import checked_csv, align_predictions\n    from src.ai_shield_full_adaptation import AUX_FEATURES, cache_auxiliary\n    lock = Path(run) / \'auxiliary_cache_hashes.json\'\n    if not lock.is_file():\n        return cache_auxiliary(aux, frames, config, run)\n    hashes = json.loads(lock.read_text())\n    tables = {}\n    for split, frame in frames.items():\n        path = Path(run) / (\'auxiliary_\' + split + \'.csv\')\n        if digest(path) != hashes[split]:\n            raise ValueError(\'Auxiliary cache changed: \' + split)\n        table = checked_csv(path)\n        if set(table.split) != {split}:\n            raise ValueError(\'Auxiliary cache has wrong split: \' + split)\n        tables[split] = align_predictions(frame, table, AUX_FEATURES)\n    print(\'Reused the complete, hash-verified auxiliary cache.\')\n    return tables\n\n\ndef main(namespace):\n    # First use saves the current completed setup; subsequent empty sessions\n    # recover it. No automatic "latest folder" guessing is used.\n    if all(key in namespace for key in (\'CONFIG\', \'RUN\', \'frames\', \'partition\')):\n        save_setup_state(namespace)\n    else:\n        restore_setup_state(namespace)\n\n\ndef prepare_notebook_setup(namespace, embedded_sources, embedded_hashes):\n    """Integrated section 3: keep this run, recover a saved setup, or start once."""\n    from src.ai_shield_adaptation import checked_csv, SPLIT_HASHES, validate_frames, EXPECTED_COUNTS, write_json\n    from src.ai_shield_full_adaptation import full_preflight, prepare_full_run\n    config = namespace[\'CONFIG\']\n    # Historical comparisons need this report on every path, including an\n    # already-loaded setup that returns without calling full_preflight again.\n    namespace[\'audit\'] = json.loads((config.audit_root / \'saved_pipeline_evaluation_report.json\').read_text(encoding=\'utf-8\'))\n    parent = config.root / \'AI_Shield_Adaptation/full_pipeline_v1\'\n    current = namespace.get(\'RUN\')\n    requested = namespace.get(\'RESUME_RUN_NAME\')\n    if current is not None:\n        selected = Path(current)\n    elif requested:\n        if not re.fullmatch(r\'\\d{8}T\\d{6}Z_[a-f0-9]{8}\', requested):\n            raise ValueError(\'Invalid RESUME_RUN_NAME\')\n        selected = parent / requested\n    elif (parent / \'active_setup.json\').is_file():\n        selected, _ = load_setup_record(config.root)\n    else:\n        # Adopt only an unambiguous existing experiment with the same settings.\n        candidates = []\n        for plan in parent.glob(\'*/experiment_plan.json\'):\n            if json.loads(plan.read_text()).get(\'config\') == asdict(config):\n                candidates.append(plan.parent)\n        if len(candidates) > 1:\n            names = \', \'.join(sorted(path.name for path in candidates))\n            raise RuntimeError(\'More than one saved experiment exists. Set RESUME_RUN_NAME in section 2 to the intended folder: \' + names)\n        selected = candidates[0] if candidates else None\n\n    if selected is not None:\n        selected = selected.resolve()\n        if selected.parent != parent.resolve() or not selected.is_dir():\n            raise RuntimeError(\'Requested experiment folder is unavailable: \' + str(selected))\n        if any((selected / name).exists() for name in (\n            \'effnet_adapted.pt\', \'semantic_adapted.pt\', \'selection_lock.json\',\n            \'frozen_full_fusion.joblib\', \'adapted_full_fusion.joblib\')):\n            raise RuntimeError(\'This experiment has progressed beyond setup. Its results are preserved; automatic optimizer/training resume is not implemented.\')\n        in_memory = (current is not None and all(key in namespace for key in (\'frames\', \'partition\')))\n        if not (selected / \'setup_resume.json\').is_file():\n            # Adopt an existing pre-training run without redoing a separate\n            # full image scan. restore_setup_state checks bytes while staging.\n            frames = {}\n            for split, expected in SPLIT_HASHES.items():\n                path = config.split_root / (split + \'.csv\')\n                if digest(path) != expected:\n                    raise ValueError(\'Original dataset manifest changed: \' + split)\n                frames[split] = checked_csv(path)\n            validate_frames(frames, EXPECTED_COUNTS)\n            adopted = dict(namespace, RUN=selected, frames=frames,\n                           partition=checked_csv(selected / \'training_roles.csv\'))\n            save_setup_state(adopted)\n        namespace[\'RESUME_RUN_NAME\'] = selected.name\n        if in_memory:\n            load_setup_record(config.root, selected.name)\n            validate_frames(namespace[\'frames\'], EXPECTED_COUNTS)\n            recorded_roles = checked_csv(selected / \'training_roles.csv\')\n            import pandas as pd\n            pd.testing.assert_frame_equal(\n                namespace[\'partition\'][[\'sample_id\', \'label\', \'role\']].reset_index(drop=True),\n                recorded_roles[[\'sample_id\', \'label\', \'role\']].reset_index(drop=True))\n            # The already completed section 3 has usable staged frames. An\n            # interrupted copy leaves Drive paths instead, so restore that case.\n            cache = Path(\'/content\') / (\'ai_shield_images_\' + selected.name)\n            all_staged = all(Path(p).parent == cache for frame in namespace[\'frames\'].values() for p in frame.local_path)\n            if all_staged:\n                namespace[\'RUN\'] = selected\n                namespace[\'cache_auxiliary\'] = reuse_complete_auxiliary\n                save_setup_state(namespace)\n                print(\'Reusing the setup already loaded in this session:\', selected.name)\n                return\n        restore_setup_state(namespace)\n        return\n\n    frames, audit = full_preflight(config)\n    run, partition = prepare_full_run(config, frames)\n    write_json(run / \'embedded_source_hashes.json\', embedded_hashes)\n    for relative, source in embedded_sources.items():\n        target = run / \'code\' / relative\n        target.parent.mkdir(parents=True, exist_ok=True)\n        target.write_text(source, encoding=\'utf-8\')\n    namespace.update(RUN=run, frames=frames, audit=audit, partition=partition)\n    # Persist before the copy: interrupted staging now has a known RUN to resume.\n    save_setup_state(namespace)\n    frames, stats = stage_verified_images(frames, Path(\'/content\') / (\'ai_shield_images_\' + run.name))\n    partition[\'local_path\'] = partition.sample_id.map(frames[\'train\'].set_index(\'sample_id\').local_path)\n    namespace.update(frames=frames, partition=partition, cache_auxiliary=reuse_complete_auxiliary)\n    print(\'Setup ready:\', stats)\n\n\nif __name__ == \'__main__\':\n    main(globals())\n', 'scripts/AI_Shield_Semantic_Compatibility_Fix_v1.py': '"""Apply to the existing Colab session; never restart, train, or relax parity.\n\nRun with exec(Path(...).read_text()) after CONFIG, frames, RUN, DEVICE exist.\nChanges are saved only after the original eight-feature GPU parity check passes.\n"""\nfrom pathlib import Path\nimport hashlib\nimport importlib\nimport json\nimport types\n\nOLD_FORWARD = \'\'\'    def forward(self, pixel_values):\n        output = self.vit(pixel_values=pixel_values, return_dict=True)\n        return self.semantic_head(output.last_hidden_state[:, 1:, :])\'\'\'\nNEW_FORWARD = \'\'\'    def forward(self, pixel_values):\n        # The saved audit used cuDNN\'s default TF32 convolution arithmetic.\n        # Scope it to ViT so CF and the other branches retain their own settings.\n        precision = torch.backends.cudnn.flags(\n            enabled=torch.backends.cudnn.enabled,\n            benchmark=torch.backends.cudnn.benchmark,\n            deterministic=torch.backends.cudnn.deterministic,\n            allow_tf32=True,\n        ) if pixel_values.device.type == "cuda" else nullcontext()\n        with precision:\n            output = self.vit(pixel_values=pixel_values, return_dict=True)\n        return self.semantic_head(output.last_hidden_state[:, 1:, :])\'\'\'\nOLD_PROCESSOR = \'AutoImageProcessor.from_pretrained(SEMANTIC_ID, revision=SEMANTIC_REVISION, use_fast=False)\'\nNEW_PROCESSOR = OLD_PROCESSOR.replace(\'use_fast=False\', \'use_fast=True\')\n\n\ndef patched_source(source):\n    for old, new in ((OLD_FORWARD, NEW_FORWARD), (OLD_PROCESSOR, NEW_PROCESSOR)):\n        if new in source:\n            continue\n        if source.count(old) != 1:\n            raise RuntimeError(\'Unrecognized source: no files changed. Share this error before continuing.\')\n        source = source.replace(old, new, 1)\n    if \'from contextlib import nullcontext\\n\' not in source:\n        if source.count(\'import math\\n\') != 1:\n            raise RuntimeError(\'Unexpected imports: no files changed.\')\n        source = source.replace(\'import math\\n\', \'import math\\nfrom contextlib import nullcontext\\n\', 1)\n    compile(source, \'semantic_compatibility_candidate\', \'exec\')\n    return source\n\n\ndef apply_fix(namespace):\n    import torch\n    import src.ai_shield_adaptation_models as current\n    from src.ai_shield_adaptation import check_audit_parity\n    required = (\'CONFIG\', \'frames\', \'RUN\', \'DEVICE\')\n    missing = [key for key in required if key not in namespace]\n    if missing:\n        raise RuntimeError(\'Run in the existing stopped session; missing: \' + \', \'.join(missing))\n    run = Path(namespace[\'RUN\'])\n    if not run.is_dir():\n        raise RuntimeError(\'The existing RUN folder is missing; no files changed.\')\n    if any((run / name).exists() for name in (\'effnet_adapted.pt\', \'semantic_adapted.pt\', \'selection_lock.json\')):\n        raise RuntimeError(\'Training artifacts already exist. Refusing to change their implementation.\')\n    if str(namespace[\'DEVICE\']).split(\':\')[0] != \'cuda\' or not torch.cuda.is_available():\n        raise RuntimeError(\'This compatibility check requires the current Colab GPU session.\')\n    if torch.cuda.get_device_capability()[0] < 8:\n        raise RuntimeError(\'This GPU lacks TF32. The saved audit used A100 arithmetic; use an A100 or L4 runtime. No files changed.\')\n\n    paths = [Path(current.__file__).resolve(), run / \'code/src/ai_shield_adaptation_models.py\']\n    plans = [(p, p.read_bytes(), patched_source(p.read_text(encoding=\'utf-8\'))) for p in paths]\n    candidate = types.ModuleType(\'semantic_compatibility_candidate\')\n    exec(compile(plans[0][2], str(paths[0]), \'exec\'), candidate.__dict__)\n    new_models, new_processor = candidate.load_original_models(namespace[\'CONFIG\'])\n    print(\'Checking the original tolerance on\', torch.cuda.get_device_name(), \'...\', flush=True)\n    report = check_audit_parity(new_models, new_processor, namespace[\'frames\'], namespace[\'CONFIG\'], namespace[\'DEVICE\'])\n\n    # Nothing on disk is changed unless every original feature passes above.\n    changes = []\n    for path, before, after in plans:\n        after_bytes = after.encode(\'utf-8\')\n        before_hash = hashlib.sha256(before).hexdigest()\n        after_hash = hashlib.sha256(after_bytes).hexdigest()\n        if before != after_bytes:\n            backup = path.with_name(path.name + \'.before-semantic-fix-\' + before_hash[:12])\n            if backup.exists() and backup.read_bytes() != before:\n                raise RuntimeError(\'Backup conflict: \' + str(backup))\n            if not backup.exists():\n                backup.write_bytes(before)\n            temporary = path.with_name(path.name + \'.semantic-fix-tmp\')\n            temporary.write_bytes(after_bytes)\n            temporary.replace(path)\n        changes.append({\'path\': str(path), \'before_sha256\': before_hash, \'after_sha256\': after_hash})\n\n    ledger = {\'fix\': \'fast_processor_and_scoped_cudnn_tf32\', \'gpu\': torch.cuda.get_device_name(),\n              \'torch\': str(torch.__version__), \'processor\': type(new_processor).__name__,\n              \'atol\': 5e-4, \'rtol\': 1e-4, \'parity\': report, \'files\': changes,\n              \'note\': \'Original embedded_source_hashes.json retained; this records the later verified patch.\'}\n    ledger_bytes = json.dumps(ledger, indent=2).encode(\'utf-8\')\n    ledger_path = run / (\'semantic_compatibility_fix_\' + hashlib.sha256(ledger_bytes).hexdigest()[:12] + \'.json\')\n    if not ledger_path.exists():\n        ledger_path.write_bytes(ledger_bytes)\n    importlib.invalidate_caches()\n    importlib.reload(current)\n    namespace.update(models=new_models, processor=new_processor, parity=report,\n                     load_original_models=current.load_original_models)\n    print(\'PASS:\', report[\'n\'], \'images; all eight original feature checks passed.\')\n    print(\'Maximum semantic logit difference:\', report[\'max_absolute_errors\'][\'semantic_vit_logit\'])\n    print(\'Backups and fix record saved. Checkpoints, images, and existing results were not changed.\')\n    if namespace.get(\'_AI_SHIELD_INTEGRATED\'):\n        print(\'Compatibility verified. Continuing with the remaining reference checks.\')\n    else:\n        print(\'Now rerun only the original failed reference-check cell, then continue below it.\')\n\n\nif __name__ == \'__main__\':\n    apply_fix(globals())\n', 'scripts/AI_Shield_Standalone_Batch_Fix_v1.py': '"""Verify fixed-size semantic inference against this trained run before saving.\n\nNo training, threshold changes, replacement predictions, or relaxed tolerances.\nThe extra semantic work is part of the measured single-request latency.\n"""\nimport ast\nimport hashlib\nimport importlib\nimport json\nfrom pathlib import Path\n\nOLD_INIT = \'    def __init__(self,models,processor,auxiliary,calibrator,fusion,device):\\n        from src.ai_shield_adaptation import PairPredictor\'\nNEW_INIT = \'\'\'    def __init__(self,models,processor,auxiliary,calibrator,fusion,device,semantic_batch_size=16):\n        from src.ai_shield_adaptation import PairPredictor\n        if isinstance(semantic_batch_size, bool) or not isinstance(semantic_batch_size, int) or semantic_batch_size < 1:\n            raise ValueError(\'semantic_batch_size must be a positive integer\')\n        self.semantic_batch_size = semantic_batch_size\'\'\'\nOLD_FORWARD = "            sem = pair.models[\'semantic\'](pair.processor(images=image,return_tensors=\'pt\')[\'pixel_values\'].to(pair.device))"\nNEW_FORWARD = \'\'\'            # Match evaluation\'s physical batch size; keep only the first result.\n            # cuDNN/ViT numerical paths can differ for a physical batch of one.\n            pixels = pair.processor(images=image,return_tensors=\'pt\')[\'pixel_values\']\n            if pixels.shape[0] != 1:\n                raise ValueError(\'Standalone inference expects one image\')\n            pixels = pixels.repeat(self.semantic_batch_size, 1, 1, 1).to(pair.device)\n            sem = pair.models[\'semantic\'](pixels)\'\'\'\nOLD_LOAD = "    return FullPredictor(models,processor,FrozenAuxiliary(config),joblib.load(Path(run)/f\'{variant}_calibrator.joblib\'),joblib.load(Path(run)/f\'{variant}_fusion.joblib\'),device)"\nNEW_LOAD = \'\'\'    saved_batch_size = int(json.loads((Path(run)/\'experiment_plan.json\').read_text())[\'config\'][\'batch_size\'])\n    return FullPredictor(models,processor,FrozenAuxiliary(config),joblib.load(Path(run)/f\'{variant}_calibrator.joblib\'),joblib.load(Path(run)/f\'{variant}_fusion.joblib\'),device,semantic_batch_size=saved_batch_size)\'\'\'\n\n\ndef patched_source(source):\n    for old, new in ((OLD_INIT, NEW_INIT), (OLD_FORWARD, NEW_FORWARD), (OLD_LOAD, NEW_LOAD)):\n        if new in source:\n            continue\n        if source.count(old) != 1:\n            raise RuntimeError(\'Unrecognized standalone source. No source files changed.\')\n        source = source.replace(old, new, 1)\n    compile(source, \'standalone_batch_candidate\', \'exec\')\n    return source\n\n\ndef check_candidate(namespace, candidate_class, batch_size):\n    import numpy as np\n    import pandas as pd\n    from src.ai_shield_adaptation import checked_csv, sha256_file\n    from src.ai_shield_full_adaptation import FULL_FEATURES\n    fusion = namespace[\'adapted_fusion\']\n\n    class Capture:\n        def predict_proba(self, values):\n            self.values = values.iloc[0].copy()\n            return fusion.predict_proba(values)\n\n    capture = Capture()\n    candidate = candidate_class(namespace[\'models\'], namespace[\'processor\'], namespace[\'auxiliary_model\'],\n        namespace[\'adapted_calibrator\'], capture, namespace[\'DEVICE\'], semantic_batch_size=batch_size)\n    test = namespace[\'frames\'][\'test\'].sort_values(\'sample_id\').groupby(\'label\', group_keys=False).head(3)\n    probes = [(\'test\', test)]\n    unseen = namespace[\'frames\'].get(\'unseen_generator\')\n    if unseen is not None and len(unseen) % batch_size:\n        probes.append((\'unseen_generator\', unseen.tail(len(unseen) % batch_size)))\n    records = []\n    for split, frame in probes:\n        saved = checked_csv(Path(namespace[\'RUN\']) / f\'adapted_full_{split}_predictions.csv\').set_index(\'sample_id\')\n        for row in frame.itertuples():\n            if sha256_file(row.local_path) != row.sha256:\n                raise ValueError(\'Reference image bytes changed: \' + row.sample_id)\n            expected = saved.loc[row.sample_id]\n            score = candidate.predict(row.local_path)\n            live = capture.values[FULL_FEATURES].to_numpy(dtype=float)\n            old = expected[FULL_FEATURES].to_numpy(dtype=float)\n            feature_match = bool(np.allclose(live, old, atol=5e-4, rtol=1e-4))\n            score_match = bool(np.isclose(score, expected.fusion_prob, atol=5e-4, rtol=1e-4))\n            records.append({\'sample_id\': row.sample_id, \'split\': split, \'saved_score\': float(expected.fusion_prob),\n                \'standalone_score\': score, \'absolute_error\': abs(score-float(expected.fusion_prob)),\n                \'max_feature_error\': float(np.max(np.abs(live-old))), \'features_match\': feature_match, \'score_matches\': score_match})\n    table = pd.DataFrame(records)\n    if \'display\' in namespace:\n        namespace[\'display\'](table)\n    if not records or not all(r[\'features_match\'] and r[\'score_matches\'] for r in records):\n        raise ValueError(\'Fixed-batch inference did not reproduce every reference. No source files were changed; share the table above.\')\n    return records\n\n\ndef apply_standalone_batch_fix(namespace):\n    import src.ai_shield_full_adaptation as current\n    required = (\'CONFIG\', \'RUN\', \'frames\', \'models\', \'processor\', \'auxiliary_model\',\n                \'adapted_calibrator\', \'adapted_fusion\', \'DEVICE\')\n    missing = [key for key in required if key not in namespace]\n    if missing:\n        raise RuntimeError(\'Keep the trained session. Missing variables: \' + \', \'.join(missing))\n    run = Path(namespace[\'RUN\'])\n    batch_size = int(json.loads((run / \'experiment_plan.json\').read_text())[\'config\'][\'batch_size\'])\n    if batch_size != namespace[\'CONFIG\'].batch_size:\n        raise ValueError(\'Current CONFIG differs from the saved evaluation batch size\')\n    current.verify_full_lock(run)\n    paths = list(dict.fromkeys([Path(current.__file__).resolve(), (run / \'code/src/ai_shield_full_adaptation.py\').resolve()]))\n    plans = [(path, path.read_bytes(), patched_source(path.read_text(encoding=\'utf-8\'))) for path in paths]\n    tree = ast.parse(plans[0][2])\n    class_node = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == \'FullPredictor\')\n    candidate_namespace = dict(vars(current))\n    exec(compile(ast.Module(body=[class_node], type_ignores=[]), \'standalone_batch_candidate\', \'exec\'), candidate_namespace)\n    print(\'Checking standalone inference with semantic internal batch size\', batch_size, \'...\', flush=True)\n    checks = check_candidate(namespace, candidate_namespace[\'FullPredictor\'], batch_size)\n\n    # Save only after all unchanged feature and score tolerances pass.\n    changes = []\n    for path, before, source in plans:\n        after = source.encode(\'utf-8\')\n        before_hash, after_hash = hashlib.sha256(before).hexdigest(), hashlib.sha256(after).hexdigest()\n        backup = path.with_name(path.name + \'.before-standalone-fix-\' + before_hash[:12])\n        if before != after:\n            if backup.exists() and backup.read_bytes() != before:\n                raise RuntimeError(\'Backup conflict: \' + str(backup))\n            if not backup.exists():\n                backup.write_bytes(before)\n            temporary = path.with_name(path.name + \'.standalone-fix-tmp\')\n            temporary.write_bytes(after)\n            temporary.replace(path)\n        changes.append({\'path\': str(path), \'before_sha256\': before_hash, \'after_sha256\': after_hash})\n    record = {\'fix\': \'fixed_semantic_batch_for_standalone\', \'logical_request_batch_size\': 1,\n              \'semantic_internal_batch_size\': batch_size, \'atol\': 5e-4, \'rtol\': 1e-4,\n              \'checks\': checks, \'files\': changes,\n              \'cost_note\': \'Repeats the input inside the semantic branch. Include this extra compute in latency and memory measurements.\'}\n    encoded = json.dumps(record, indent=2).encode(\'utf-8\')\n    report_path = run / (\'standalone_batch_compatibility_\' + hashlib.sha256(encoded).hexdigest()[:12] + \'.json\')\n    if not report_path.exists():\n        report_path.write_bytes(encoded)\n    importlib.invalidate_caches()\n    importlib.reload(current)\n    namespace[\'FullPredictor\'] = current.FullPredictor\n    namespace[\'load_full_predictor\'] = current.load_full_predictor\n    namespace[\'semantic_internal_batch_size\'] = batch_size\n    print(\'PASS:\', len(checks), \'reference images; feature and score checks passed at the original tolerance.\')\n    print(\'Weights, thresholds, saved predictions and the selection lock were not changed.\')\n    print(\'Continuing section 8. Timing includes the repeated semantic computation.\')\n    return checks\n\n\nif __name__ == \'__main__\':\n    apply_standalone_batch_fix(globals())\n', 'src/__init__.py': ''}
EMBEDDED_HASHES = {'src/ai_shield_adaptation.py': 'd10eb8cb82247a413d3d27c7bd8a1ababbe9129858bbae3164adfa79c39f4410', 'src/ai_shield_adaptation_models.py': '4651eed117e378d5afd01c2715861ce8f66c0e64346f5a6ece4a32aa94e47e1d', 'src/ai_shield_metrics.py': 'd4f40c0ee1a152f8e6453e5206696e9ebc19107b91f9a354ad7d1008eb8de2d2', 'src/ai_shield_cf_inference.py': '12b03eeade84fe7027ad70cc2782fb85dd857ec2e681e764b8b936f9b3725c52', 'src/ai_shield_full_adaptation.py': '6e119146b9ae7383e0de745bfdd3a4fa36fd065769d6b1931edf85045a28f6c7', 'scripts/AI_Shield_Setup_Resume_v1.py': '9214c03ade7dc95e344863b307e129f8948f094d6d796cafc4d1a9fbbc9a4040', 'scripts/AI_Shield_Semantic_Compatibility_Fix_v1.py': 'fa2b50a90382a1a65a89baa7fbd2f7ceb61f0ddd685361aaa94f2bf15daa957e', 'scripts/AI_Shield_Standalone_Batch_Fix_v1.py': '46c1707178ceadefc570cc3f311804248f21ddd9886d8a4afe2c047d0f9ce84a', 'src/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855'}

import sys, hashlib, os, json, importlib, runpy
from pathlib import Path
restart_marker = Path('/content/ai_shield_restart_required.json')
if restart_marker.exists() and json.loads(restart_marker.read_text()).get('pid') == os.getpid():
    raise RuntimeError('Setup succeeded. Restart the Colab session once, then rerun from section 2 to load the installed libraries.')
CODE_ROOT = Path('/content/ai_shield_full_adaptation_code')
for relative, source in EMBEDDED_SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == EMBEDDED_HASHES[relative]
    path = CODE_ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
sys.path.insert(0, str(CODE_ROOT))
# Refresh our Python modules when this cell is rerun in an existing session.
# Existing frames/RUN/partition stay in the notebook namespace.
for key in list(sys.modules):
    if key == 'src' or key.startswith('src.'):
        del sys.modules[key]
importlib.invalidate_caches()
setup_tools = runpy.run_path(str(CODE_ROOT / 'scripts/AI_Shield_Setup_Resume_v1.py'))
compatibility_tools = runpy.run_path(str(CODE_ROOT / 'scripts/AI_Shield_Semantic_Compatibility_Fix_v1.py'))
_AI_SHIELD_INTEGRATED = True
print('Integrated fixes v2 loaded. Continue with the next cell.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, platform, time
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from src.ai_shield_adaptation import (
    seed_everything, check_audit_parity, train_branch, write_json,
    historical_references, checked_csv, align_predictions, benchmark, sha256_file)
from src.ai_shield_adaptation_models import load_original_models
from src.ai_shield_full_adaptation import *

CONFIG = globals().get('CONFIG', FullConfig())
# Normally leave this alone: current RUN or the saved setup is reused.
RESUME_RUN_NAME = globals().get('RESUME_RUN_NAME', None)
STAGE_TO_LOCAL = True  # Faster training: copy verified image bytes to temporary runtime storage.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then reconnect.')
if torch.cuda.get_device_capability()[0] < 8:
    raise RuntimeError('The original semantic audit used TF32 on A100. Select A100 or L4 to reproduce those checks; this GPU lacks TF32.')
seed_everything(CONFIG.seed)
print('GPU:', torch.cuda.get_device_name(0))
print('Per branch:', CONFIG.head_epochs, 'head epochs +', CONFIG.partial_epochs, 'limited-backbone epochs')


## 3. Verify the saved data and reserve three training roles
This cell now saves setup automatically and reuses the current or saved experiment.
On recovery, surviving local images are reused after hashing; missing images are copied and
verified in a single pass. No separate repair or resume script needs to be pasted into the notebook.
If multiple older experiments exist with no saved selection, set RESUME_RUN_NAME in section 2.
The complete dataset contains **5,124 images**: 2,990 train, 640 validation, 640 test,
and 854 in the GPT Image 2 held-out-generator comparison. These are your existing splits.

Generated images come from QwenImageBench; real images come from Open Images V7.
The five development generators are FLUX.2 max, GLM-Image, Qwen-Image-2.0-pro,
Seedream-5.0 and nano-banana-2.0. GPT Image 2 remains excluded from fitting.
"Qwen data" here means the multi-generator benchmark, not only Qwen-generated images.

Within the 2,990 training images, roughly **70% train the networks, 15% calibrate scores,
and 15% fit fusion**. Connected prompts, duplicates and source groups stay together,
so the printed counts can differ from those percentages. Validation chooses checkpoints
and thresholds. Test images never fit a model, calibration or threshold.

CF used all 2,990 training images for its own network. This matches the overall data pool,
**not per-network exposure, training compute or earlier pretraining costs**.
The tests have already been inspected in earlier work: they are historical comparisons,
not a new blind confirmation set.


In [ ]:
setup_tools['prepare_notebook_setup'](globals(), EMBEDDED_SOURCES, EMBEDDED_HASHES)
display(partition.groupby(['role', 'label']).size().rename('images').reset_index())
display(pd.concat(frames.values()).groupby(['split', 'generator', 'label']).size().rename('images').reset_index())
print('Setup ready. Results folder:', RUN)


## 4. Restore the original system and check it
Before training, reproduce the main branches' saved features on six reference images,
replay the original 17-feature fusion on both historical tests, and check face/no-face cases.
The fusion replay uses the original cached features; it is not an assertion that every live
auxiliary output is bit-identical to the old notebook.

The VAE originally sampled a random latent vector even in evaluation mode. We preserve
that sampled architecture but now seed **one CPU sample by the image's SHA-256** on every split.
This makes new evaluation and standalone predictions agree on the same bytes.
Both refitted variants share these exact frozen features. The original saved reference
keeps its old draws and old thresholds; it also differs in calibration and fusion.
The frozen-versus-fine-tuned comparison isolates the effect of network adaptation.

The existing image preprocessing is preserved, including no EXIF transpose in the old branches.
CF retains its own preprocessing. The face gate still decides when OpenForensics is applicable;
OpenForensics receives the full resized image, as in your original audited implementation.
Any parity failure stops the experiment so it can be investigated before training.
The integrated compatibility check uses the fast image processor and scopes cuDNN TF32
to the ViT backbone. The original tolerances are unchanged. Only a passing live GPU check
can update an older saved implementation; backup files and a change record are retained.


In [ ]:
compatibility_tools['apply_fix'](globals())
setup_tools['save_setup_state'](globals())
# Keep earlier reports when these small live checks are repeated after recovery.
check_reports = RUN
if any((RUN / name).exists() for name in ('main_branch_parity.json', 'original_full_fusion_replay.json', 'face_parity.json')):
    check_reports = RUN / 'resume_checks' / uuid.uuid4().hex[:12]
    check_reports.mkdir(parents=True, exist_ok=False)
write_json(check_reports / 'main_branch_parity.json', parity)
replay = replay_original(CONFIG, frames, check_reports)
auxiliary_model = FrozenAuxiliary(CONFIG)
face_parity = verify_face_parity(auxiliary_model, CONFIG, frames, check_reports)
print('Reference checks passed:', parity['n'], 'main-branch images;', len(face_parity), 'face/no-face images.')
display(pd.DataFrame(replay))


In [ ]:
cache_started = time.perf_counter()
auxiliary = cache_auxiliary(auxiliary_model, frames, CONFIG, RUN)
auxiliary_seconds = time.perf_counter() - cache_started
print('Frozen auxiliary features cached for all', sum(map(len, frames.values())), 'images.')
print('Seconds:', round(auxiliary_seconds, 1))


## 5. Control experiment: keep the networks frozen and refit the combination
Refit the EfficientNet calibration on its reserved images, then the full 17-feature logistic
fusion on a different reserved subset. This measures how much improvement comes from
adapting the score combination alone. Each output's **max-F1 threshold** is selected on validation.
F1 balances generated-image recall and precision; also inspect the real-image false-positive rate.
Scores are detector outputs, not demonstrated probabilities that an image is AI-generated.


In [ ]:
frozen_calibrator, frozen_fusion, frozen_thresholds, frozen_selection = fit_full_variant(
    'frozen_full', models, processor, partition, frames, auxiliary, CONFIG, RUN, DEVICE)
display(pd.DataFrame([frozen_selection['validation_metrics']]))


## 6. Fine-tune the two main branches
Each starts from your saved model. First train the head for three epochs, then allow the
final backbone block to learn for three epochs at a smaller learning rate. Earlier blocks
and their BatchNorm statistics remain frozen. VAE and OpenForensics are never trained.
Each branch's best weights are selected by validation AUROC, with the original weights
also eligible. If adaptation does not help validation, the selected weights can remain original.
Training time, trainable parameter counts, validation history and checkpoint hashes are saved.

A disconnected run retains its saved files. This first version does not automatically resume
optimizer state: use a new run after an interrupted training stage, rather than mixing results.


In [ ]:
training_reports = []
branch_train = partition[partition.role == 'branch_train'].copy()
for branch, model in models.items():
    report = train_branch(model, branch, processor, branch_train,
        frames['validation'], CONFIG, RUN, DEVICE)
    training_reports.append(report)
    print('Selected checkpoint saved for:', branch)
display(pd.DataFrame(training_reports))


In [ ]:
adapted_calibrator, adapted_fusion, adapted_thresholds, adapted_selection = fit_full_variant(
    'adapted_full', models, processor, partition, frames, auxiliary, CONFIG, RUN, DEVICE)
selections = {'frozen_full': frozen_selection, 'adapted_full': adapted_selection}
display(pd.DataFrame([{'variant': name, **value['validation_metrics']}
                      for name, value in selections.items()]))


## 7. Lock the choices, then evaluate
The preferred new candidate is selected by validation F1, then validation AUROC.
Files and thresholds are locked before producing test metrics. The test results can inform
your report but cannot be used to pick a different checkpoint, threshold or candidate in this run.


In [ ]:
# Read reporting metadata explicitly; it must not depend on which setup path ran.
audit = json.loads((CONFIG.audit_root / 'saved_pipeline_evaluation_report.json').read_text(encoding='utf-8'))
selection_lock = lock_full_selection(RUN, selections)
verify_full_lock(RUN)
verify_input_lock(CONFIG, RUN)
print('Validation-selected candidate:', selection_lock['validation_selected_candidate'])
comparison_rows, generator_rows = evaluate_full_variant('adapted_full', models, processor,
    frames, auxiliary, adapted_calibrator, adapted_fusion, adapted_thresholds, CONFIG, RUN, DEVICE)
original_models, original_processor = load_original_models(CONFIG)
rows, subgroups = evaluate_full_variant('frozen_full', original_models, original_processor,
    frames, auxiliary, frozen_calibrator, frozen_fusion, frozen_thresholds, CONFIG, RUN, DEVICE)
comparison_rows += rows
generator_rows += subgroups
del original_models, original_processor
gc.collect()
torch.cuda.empty_cache()
comparison_rows += historical_references(frames, CONFIG, audit, RUN)
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RUN / 'comparison.csv', index=False)
display(comparison[['model', 'split', 'auc', 'f1', 'accuracy', 'fpr', 'threshold']])
display(pd.DataFrame(generator_rows))


## 8. Check standalone predictions and measure speed against CF
First check that single-image inference agrees with the batched adapted results on six fixed
test images. This checks implementation only; it does not select or train the model.
Then time 64 fixed validation images on this same runtime: three warmups, three timed passes,
batch size one. The full pipeline includes live VAE, face checks, file reading and fusion.
Timing excludes loading weights, the API and provenance. Process RAM is not isolated model memory.
The semantic branch uses the saved evaluation batch size internally by repeating the one uploaded
image and keeping one result. This avoids the observed singleton numerical discrepancy. Its extra
computation is included in timing and explicitly recorded; one request still contains one image.
The first cell includes a guarded repair for an already-trained session. It checks the six reference
images and the last partial unseen-generator batch before saving any inference-code change.


In [ ]:
# Standalone batch compatibility fix: no retraining; original tolerances.
exec('"""Verify fixed-size semantic inference against this trained run before saving.\n\nNo training, threshold changes, replacement predictions, or relaxed tolerances.\nThe extra semantic work is part of the measured single-request latency.\n"""\nimport ast\nimport hashlib\nimport importlib\nimport json\nfrom pathlib import Path\n\nOLD_INIT = \'    def __init__(self,models,processor,auxiliary,calibrator,fusion,device):\\n        from src.ai_shield_adaptation import PairPredictor\'\nNEW_INIT = \'\'\'    def __init__(self,models,processor,auxiliary,calibrator,fusion,device,semantic_batch_size=16):\n        from src.ai_shield_adaptation import PairPredictor\n        if isinstance(semantic_batch_size, bool) or not isinstance(semantic_batch_size, int) or semantic_batch_size < 1:\n            raise ValueError(\'semantic_batch_size must be a positive integer\')\n        self.semantic_batch_size = semantic_batch_size\'\'\'\nOLD_FORWARD = "            sem = pair.models[\'semantic\'](pair.processor(images=image,return_tensors=\'pt\')[\'pixel_values\'].to(pair.device))"\nNEW_FORWARD = \'\'\'            # Match evaluation\'s physical batch size; keep only the first result.\n            # cuDNN/ViT numerical paths can differ for a physical batch of one.\n            pixels = pair.processor(images=image,return_tensors=\'pt\')[\'pixel_values\']\n            if pixels.shape[0] != 1:\n                raise ValueError(\'Standalone inference expects one image\')\n            pixels = pixels.repeat(self.semantic_batch_size, 1, 1, 1).to(pair.device)\n            sem = pair.models[\'semantic\'](pixels)\'\'\'\nOLD_LOAD = "    return FullPredictor(models,processor,FrozenAuxiliary(config),joblib.load(Path(run)/f\'{variant}_calibrator.joblib\'),joblib.load(Path(run)/f\'{variant}_fusion.joblib\'),device)"\nNEW_LOAD = \'\'\'    saved_batch_size = int(json.loads((Path(run)/\'experiment_plan.json\').read_text())[\'config\'][\'batch_size\'])\n    return FullPredictor(models,processor,FrozenAuxiliary(config),joblib.load(Path(run)/f\'{variant}_calibrator.joblib\'),joblib.load(Path(run)/f\'{variant}_fusion.joblib\'),device,semantic_batch_size=saved_batch_size)\'\'\'\n\n\ndef patched_source(source):\n    for old, new in ((OLD_INIT, NEW_INIT), (OLD_FORWARD, NEW_FORWARD), (OLD_LOAD, NEW_LOAD)):\n        if new in source:\n            continue\n        if source.count(old) != 1:\n            raise RuntimeError(\'Unrecognized standalone source. No source files changed.\')\n        source = source.replace(old, new, 1)\n    compile(source, \'standalone_batch_candidate\', \'exec\')\n    return source\n\n\ndef check_candidate(namespace, candidate_class, batch_size):\n    import numpy as np\n    import pandas as pd\n    from src.ai_shield_adaptation import checked_csv, sha256_file\n    from src.ai_shield_full_adaptation import FULL_FEATURES\n    fusion = namespace[\'adapted_fusion\']\n\n    class Capture:\n        def predict_proba(self, values):\n            self.values = values.iloc[0].copy()\n            return fusion.predict_proba(values)\n\n    capture = Capture()\n    candidate = candidate_class(namespace[\'models\'], namespace[\'processor\'], namespace[\'auxiliary_model\'],\n        namespace[\'adapted_calibrator\'], capture, namespace[\'DEVICE\'], semantic_batch_size=batch_size)\n    test = namespace[\'frames\'][\'test\'].sort_values(\'sample_id\').groupby(\'label\', group_keys=False).head(3)\n    probes = [(\'test\', test)]\n    unseen = namespace[\'frames\'].get(\'unseen_generator\')\n    if unseen is not None and len(unseen) % batch_size:\n        probes.append((\'unseen_generator\', unseen.tail(len(unseen) % batch_size)))\n    records = []\n    for split, frame in probes:\n        saved = checked_csv(Path(namespace[\'RUN\']) / f\'adapted_full_{split}_predictions.csv\').set_index(\'sample_id\')\n        for row in frame.itertuples():\n            if sha256_file(row.local_path) != row.sha256:\n                raise ValueError(\'Reference image bytes changed: \' + row.sample_id)\n            expected = saved.loc[row.sample_id]\n            score = candidate.predict(row.local_path)\n            live = capture.values[FULL_FEATURES].to_numpy(dtype=float)\n            old = expected[FULL_FEATURES].to_numpy(dtype=float)\n            feature_match = bool(np.allclose(live, old, atol=5e-4, rtol=1e-4))\n            score_match = bool(np.isclose(score, expected.fusion_prob, atol=5e-4, rtol=1e-4))\n            records.append({\'sample_id\': row.sample_id, \'split\': split, \'saved_score\': float(expected.fusion_prob),\n                \'standalone_score\': score, \'absolute_error\': abs(score-float(expected.fusion_prob)),\n                \'max_feature_error\': float(np.max(np.abs(live-old))), \'features_match\': feature_match, \'score_matches\': score_match})\n    table = pd.DataFrame(records)\n    if \'display\' in namespace:\n        namespace[\'display\'](table)\n    if not records or not all(r[\'features_match\'] and r[\'score_matches\'] for r in records):\n        raise ValueError(\'Fixed-batch inference did not reproduce every reference. No source files were changed; share the table above.\')\n    return records\n\n\ndef apply_standalone_batch_fix(namespace):\n    import src.ai_shield_full_adaptation as current\n    required = (\'CONFIG\', \'RUN\', \'frames\', \'models\', \'processor\', \'auxiliary_model\',\n                \'adapted_calibrator\', \'adapted_fusion\', \'DEVICE\')\n    missing = [key for key in required if key not in namespace]\n    if missing:\n        raise RuntimeError(\'Keep the trained session. Missing variables: \' + \', \'.join(missing))\n    run = Path(namespace[\'RUN\'])\n    batch_size = int(json.loads((run / \'experiment_plan.json\').read_text())[\'config\'][\'batch_size\'])\n    if batch_size != namespace[\'CONFIG\'].batch_size:\n        raise ValueError(\'Current CONFIG differs from the saved evaluation batch size\')\n    current.verify_full_lock(run)\n    paths = list(dict.fromkeys([Path(current.__file__).resolve(), (run / \'code/src/ai_shield_full_adaptation.py\').resolve()]))\n    plans = [(path, path.read_bytes(), patched_source(path.read_text(encoding=\'utf-8\'))) for path in paths]\n    tree = ast.parse(plans[0][2])\n    class_node = next(node for node in tree.body if isinstance(node, ast.ClassDef) and node.name == \'FullPredictor\')\n    candidate_namespace = dict(vars(current))\n    exec(compile(ast.Module(body=[class_node], type_ignores=[]), \'standalone_batch_candidate\', \'exec\'), candidate_namespace)\n    print(\'Checking standalone inference with semantic internal batch size\', batch_size, \'...\', flush=True)\n    checks = check_candidate(namespace, candidate_namespace[\'FullPredictor\'], batch_size)\n\n    # Save only after all unchanged feature and score tolerances pass.\n    changes = []\n    for path, before, source in plans:\n        after = source.encode(\'utf-8\')\n        before_hash, after_hash = hashlib.sha256(before).hexdigest(), hashlib.sha256(after).hexdigest()\n        backup = path.with_name(path.name + \'.before-standalone-fix-\' + before_hash[:12])\n        if before != after:\n            if backup.exists() and backup.read_bytes() != before:\n                raise RuntimeError(\'Backup conflict: \' + str(backup))\n            if not backup.exists():\n                backup.write_bytes(before)\n            temporary = path.with_name(path.name + \'.standalone-fix-tmp\')\n            temporary.write_bytes(after)\n            temporary.replace(path)\n        changes.append({\'path\': str(path), \'before_sha256\': before_hash, \'after_sha256\': after_hash})\n    record = {\'fix\': \'fixed_semantic_batch_for_standalone\', \'logical_request_batch_size\': 1,\n              \'semantic_internal_batch_size\': batch_size, \'atol\': 5e-4, \'rtol\': 1e-4,\n              \'checks\': checks, \'files\': changes,\n              \'cost_note\': \'Repeats the input inside the semantic branch. Include this extra compute in latency and memory measurements.\'}\n    encoded = json.dumps(record, indent=2).encode(\'utf-8\')\n    report_path = run / (\'standalone_batch_compatibility_\' + hashlib.sha256(encoded).hexdigest()[:12] + \'.json\')\n    if not report_path.exists():\n        report_path.write_bytes(encoded)\n    importlib.invalidate_caches()\n    importlib.reload(current)\n    namespace[\'FullPredictor\'] = current.FullPredictor\n    namespace[\'load_full_predictor\'] = current.load_full_predictor\n    namespace[\'semantic_internal_batch_size\'] = batch_size\n    print(\'PASS:\', len(checks), \'reference images; feature and score checks passed at the original tolerance.\')\n    print(\'Weights, thresholds, saved predictions and the selection lock were not changed.\')\n    print(\'Continuing section 8. Timing includes the repeated semantic computation.\')\n    return checks\n\n\nif __name__ == \'__main__\':\n    apply_standalone_batch_fix(globals())\n')

predictor = FullPredictor(models, processor, auxiliary_model, adapted_calibrator, adapted_fusion, DEVICE,
    semantic_batch_size=CONFIG.batch_size)
reference_images = frames['test'].sort_values('sample_id').groupby('label', group_keys=False).head(3)
saved = checked_csv(RUN / 'adapted_full_test_predictions.csv').set_index('sample_id')
standalone_checks = []
for row in reference_images.itertuples():
    actual = predictor.predict(row.local_path)
    expected = float(saved.loc[row.sample_id, 'fusion_prob'])
    if not np.isclose(actual, expected, atol=5e-4, rtol=1e-4):
        raise ValueError('Standalone prediction differs: ' + row.sample_id)
    standalone_checks.append({'sample_id': row.sample_id, 'batch_score': expected,
                              'standalone_score': actual, 'absolute_error': abs(actual - expected)})
write_json(RUN / 'standalone_inference_parity.json', standalone_checks)
display(pd.DataFrame(standalone_checks))

timing_images = frames['validation'].sort_values('sample_id').groupby('label', group_keys=False).head(32)
paths = timing_images.local_path.tolist()
timing = {'device': DEVICE, 'gpu': torch.cuda.get_device_name(0), 'platform': platform.platform(),
          'torch': torch.__version__, 'cpu_threads': torch.get_num_threads(),
          'sample_ids': timing_images.sample_id.tolist()}
timing['adapted_full'] = benchmark(predictor.predict, paths, DEVICE)
timing['adapted_full']['semantic_internal_batch_size'] = predictor.semantic_batch_size
timing['adapted_full']['logical_request_batch_size'] = 1
timing['adapted_full']['batch_note'] = 'The semantic branch repeats the input internally; all repeated computation is included in the measured latency and memory.'
for model in models.values():
    model.cpu()
del predictor
gc.collect()
torch.cuda.empty_cache()


In [ ]:
from src.ai_shield_cf_inference import FrozenCF
cf = FrozenCF(CONFIG.cf_root / 'checkpoints/stage2_last_blocks/best.pt',
    expected_sha256='51532d28870706a3c380a29889f634c556583bd62ed74775e5e610d5fd9d7b5f',
    threshold=0.6339887976646423,
    expected_manifest_sha256='2ba189a3be70e6840b6be6b530253925e87d7782c2882f0189ce8e582f20d01d',
    device=DEVICE)
cf_live = pd.DataFrame([{'sample_id': row.sample_id, 'label': row.label,
    **cf.predict(row.local_path)} for row in timing_images.itertuples()])
cf_saved = checked_csv(CONFIG.cf_root / 'predictions/finetuned_validation.csv')
cf_saved = align_predictions(timing_images, cf_saved[cf_saved.sample_id.isin(timing_images.sample_id)], ['prob_fake'])
if not np.allclose(cf_live.prob_fake, cf_saved.prob_fake, atol=1e-5, rtol=1e-5):
    raise ValueError('CF runtime parity failed')
timing['fine_tuned_cf'] = benchmark(cf.predict, paths, DEVICE)
write_json(RUN / 'runtime_comparison.json', timing)
del cf
gc.collect()
torch.cuda.empty_cache()
display(pd.DataFrame([{'model': name, **timing[name]} for name in ('adapted_full', 'fine_tuned_cf')]))


## 9. Save the report and interpret it honestly
- **Fine-tuned vs frozen/refitted:** did adapting the main networks add value?
- **Full fusion vs each main branch:** did combining signals help or hurt?
- **Full pipeline vs CF:** compare F1, AUROC, real-image false alarms, per-generator recall and latency.
- A lower-cost or faster method can still be useful; small differences need repeat runs and uncertainty estimates.

These splits have source/format shortcuts: generated images are square, and the GPT Image 2
set is PNG while its real comparison is JPEG. Repeated prompts also mean images are not all
independent examples. This notebook does not prove those shortcuts have been removed.
Next checks are matched JPEG/resize conditions, human-created artwork/renders/edited photos,
and a fresh generator/content set that has not guided development. Do not claim general image
authenticity or replacement of CF from one good table.


In [ ]:
verify_full_lock(RUN)
verify_input_lock(CONFIG, RUN)
report = {'status': 'completed', 'run': str(RUN), 'scope': 'full_17_feature_pipeline',
    'validation_selected_candidate': selection_lock['validation_selected_candidate'],
    'training': training_reports, 'comparison': comparison_rows, 'generator_breakdown': generator_rows,
    'main_branch_parity': parity, 'original_fusion_replay': replay, 'face_parity': face_parity,
    'standalone_parity': standalone_checks, 'auxiliary_cache_seconds': auxiliary_seconds,
    'runtime': timing, 'selection_lock': selection_lock,
    'caveats': ['Previously inspected historical tests; fresh confirmation remains necessary.',
                'Same overall training pool as CF, not identical per-network exposure or compute.',
                'New VAE draws are deterministic per image; the original reference retains its old draws.',
                'Format/source/content shortcuts and non-photographic real images still need dedicated tests.',
                'No demonstrated probability calibration or isolated peak RAM measurement.']}
write_json(RUN / 'final_report.json', report)
print('Done. Read comparison.csv and final_report.json in:')
print(RUN)
print('Selected research candidate:', report['validation_selected_candidate'])


## Optional: try an uploaded image after the run
This cell defaults to **off** so Run all completes without waiting for an upload.
Turn it on and rerun this cell to try the validation-selected candidate.
A score above its saved threshold is classified as generated; a score is not a certainty.


In [ ]:
TRY_UPLOAD = False
if TRY_UPLOAD:
    from google.colab import files
    variant = selection_lock['validation_selected_candidate']
    preview = load_full_predictor(CONFIG, RUN, DEVICE, variant=variant)
    threshold = selections[variant]['thresholds']['fusion_prob']
    for name, content in files.upload().items():
        path = Path('/content') / Path(name).name
        path.write_bytes(content)
        score = preview.predict(path)
        print(name, {'model': variant, 'generated_score': round(score, 4),
                     'threshold': threshold, 'label': 'generated' if score >= threshold else 'real'})
    del preview
    torch.cuda.empty_cache()


## Reuse and sources
Results live under `MyDrive/AI_Shield_Adaptation/full_pipeline_v1/<run_id>/`.
Saved files include both fine-tuned checkpoints, each variant's calibration/fusion,
thresholds, split roles, feature caches, per-image predictions, training histories and the final report.
The `code/` folder contains the exact implementation. To reuse a **completed** run after installing
dependencies and mounting Drive, add its `code` directory to `sys.path`, then call:
```python
from src.ai_shield_full_adaptation import FullConfig, load_full_predictor
predictor = load_full_predictor(FullConfig(), run_path, device='cuda', variant='adapted_full')
score = predictor.predict(image_path)
```
Use the variant and threshold recorded in `selection_lock.json` and its selection JSON.
Original Drive artifacts remain required for loading the frozen auxiliary models.

- [Original saved-pipeline audit](https://colab.research.google.com/drive/10jl7g0qknxLUCGF1VddbqbVf43oef4oD)
- [Completed CF fine-tuning](https://colab.research.google.com/drive/1q5CB7k9fdY9ylfqMDQgClVl4k_ogNJef)
- [Stacking and separate fitting data](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)
- [PyTorch evaluation and gradient modes](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [Pinned TensorFlow release, including Python 3.13](https://pypi.org/project/tensorflow-cpu/2.20.0/)
- [OpenCV package compatibility](https://pypi.org/project/opencv-python-headless/4.11.0.86/)
